In [ ]:
!pip install statsbombpy duckdb

import os, warnings
import pandas as pd
from statsbombpy import sb
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
os.makedirs(f"{folder}/parts", exist_ok=True)

season = "2015/2016"
leagues = {
    "seriea": "Serie A",
    "premier": "Premier League",
    "laliga": "La Liga",
    "ligue1": "Ligue 1",
}

keep = [
    "match_id", "team", "player", "player_id", "position", "type",
    "period", "minute", "second", "location", "under_pressure",
    "pass_end_location", "pass_outcome", "pass_length", "pass_cross",
    "pass_through_ball", "pass_shot_assist", "pass_goal_assist",
    "shot_outcome", "shot_statsbomb_xg", "dribble_outcome",
    "duel_type", "duel_outcome", "interception_outcome",
    "carry_end_location", "tactics", "substitution_replacement",
]

comps = sb.competitions()

def load_league(key, name, chunk=100):
    row = comps[(comps.competition_name == name) & (comps.season_name == season)].iloc[0]
    matches = sb.matches(competition_id=row.competition_id, season_id=row.season_id)
    matches.to_pickle(f"{folder}/{key}_1516_matches.pkl")
    ids = list(matches.match_id)
    parts = []
    for start in range(0, len(ids), chunk):
        part_path = f"{folder}/parts/{key}_{start}.pkl"
        if os.path.exists(part_path):
            parts.append(pd.read_pickle(part_path))
            continue
        frames = []
        for mid in ids[start:start + chunk]:
            try:
                e = sb.events(match_id=mid)
            except Exception as ex:
                print("ошибка в матче", mid, ex)
                continue
            frames.append(e[[c for c in keep if c in e.columns]])
        df = pd.concat(frames, ignore_index=True)
        df.to_pickle(part_path)
        parts.append(df)
        print(name, min(start + chunk, len(ids)), "из", len(ids))
    out = pd.concat(parts, ignore_index=True)
    out.to_pickle(f"{folder}/{key}_1516_events.pkl")
    return out.shape

for key, name in leagues.items():
    final = f"{folder}/{key}_1516_events.pkl"
    if os.path.exists(final):
        print(name, "уже есть:", pd.read_pickle(final).shape)
    else:
        print(name, "скачано:", load_league(key, name))

In [ ]:
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

leagues = {
    "seriea": "Serie A",
    "premier": "Premier League",
    "laliga": "La Liga",
    "ligue1": "Ligue 1",
}

def role(p):
    if p == "Goalkeeper": return "GK"
    if "Center Back" in p: return "CB"
    if "Back" in p: return "FB"            # Left/Right Back, Wing Back
    if "Defensive Midfield" in p: return "DM"
    if "Attacking Midfield" in p: return "AM_W"
    if "Midfield" in p: return "CM"
    if "Wing" in p: return "AM_W"
    return "ST"

def build_league(key, name):
    events = pd.read_pickle(f"{folder}/{key}_1516_events.pkl")

    # 1. минуты, сыгранные каждым игроком
    match_end = events.groupby("match_id")["minute"].max()
    stint = {}
    for _, r in events[events.type == "Starting XI"].iterrows():
        for p in r.tactics["lineup"]:
            stint[(r.match_id, p["player"]["name"])] = [0, match_end[r.match_id], r.team]
    for _, r in events[events.type == "Substitution"].iterrows():
        mid = r.match_id
        if (mid, r.player) in stint:
            stint[(mid, r.player)][1] = r.minute
        stint[(mid, r.substitution_replacement)] = [r.minute, match_end[mid], r.team]
    minutes = pd.DataFrame(
        [(m, p, t, s, e) for (m, p), (s, e, t) in stint.items()],
        columns=["match_id", "player", "team", "start", "end"],
    )
    minutes["mins"] = minutes["end"] - minutes["start"]

    # 2. координаты
    def split_xy(col, prefix):
        events[prefix + "_x"] = np.nan
        events[prefix + "_y"] = np.nan
        s = events[col]
        ok = s.notna()
        arr = pd.DataFrame(s[ok].tolist(), index=s[ok].index)
        events.loc[ok, prefix + "_x"] = arr[0]
        events.loc[ok, prefix + "_y"] = arr[1]

    split_xy("location", "s")
    split_xy("pass_end_location", "pe")
    split_xy("carry_end_location", "ce")

    def dgoal(x, y):
        return np.sqrt((120 - x) ** 2 + (40 - y) ** 2)

    events["pass_prog"] = dgoal(events.s_x, events.s_y) - dgoal(events.pe_x, events.pe_y)
    events["carry_prog"] = dgoal(events.s_x, events.s_y) - dgoal(events.ce_x, events.ce_y)

    # 3. отметки действий
    t = events.type
    done = events.pass_outcome.isna()
    won = ["Won", "Success In Play", "Success Out"]
    flags = pd.DataFrame({
        "passes": t == "Pass",
        "passes_done": (t == "Pass") & done,
        "prog_passes": (t == "Pass") & done & (events.pass_prog >= 10),
        "long_passes": (t == "Pass") & done & (events.pass_length >= 30),
        "final_third_passes": (t == "Pass") & done & (events.pe_x >= 80),
        "key_passes": (events.pass_shot_assist == True) | (events.pass_goal_assist == True),
        "assists": events.pass_goal_assist == True,
        "crosses": events.pass_cross == True,
        "through_balls": events.pass_through_ball == True,
        "shots": t == "Shot",
        "goals": (t == "Shot") & (events.shot_outcome == "Goal"),
        "dribbles": t == "Dribble",
        "dribbles_won": (t == "Dribble") & (events.dribble_outcome == "Complete"),
        "carries": t == "Carry",
        "prog_carries": (t == "Carry") & (events.carry_prog >= 10),
        "receipts": t == "Ball Receipt*",
        "tackles": (t == "Duel") & (events.duel_type == "Tackle"),
        "tackles_won": (t == "Duel") & (events.duel_type == "Tackle") & events.duel_outcome.isin(won),
        "aerial_lost": (t == "Duel") & (events.duel_type == "Aerial Lost"),
        "interceptions": t == "Interception",
        "recoveries": t == "Ball Recovery",
        "pressures": t == "Pressure",
        "clearances": t == "Clearance",
        "blocks": t == "Block",
        "fouls": t == "Foul Committed",
        "fouls_won": t == "Foul Won",
        "dispossessed": t == "Dispossessed",
    })
    flags["xg"] = events.shot_statsbomb_xg.fillna(0)
    flags["player"] = events.player
    stats = flags[flags.player.notna()].groupby("player").sum()

    # 4. позиция, id, средняя точка действий, команда, минуты
    named = events.dropna(subset=["player"])
    pos = named.dropna(subset=["position"]).groupby("player")["position"].agg(lambda s: s.mode().iat[0])
    pid = named.groupby("player")["player_id"].agg(lambda s: s.mode().iat[0])
    avg = named.dropna(subset=["s_x"]).groupby("player")[["s_x", "s_y"]].mean()
    pm = minutes.groupby("player")["mins"].sum()
    pt = minutes.groupby(["player", "team"])["mins"].sum().reset_index().sort_values("mins", ascending=False)
    team = pt.drop_duplicates("player").set_index("player")["team"]

    players = (stats.join(pm.rename("mins")).join(pos.rename("position"))
               .join(pid.rename("player_id")).join(avg).join(team.rename("team")))
    players["role"] = players.position.map(role)
    players = players[(players.mins >= 900) & (players.role != "GK")]

    # 5. на 90 минут
    count_cols = [c for c in flags.columns if c != "player"]
    per90 = players.copy()
    for c in count_cols:
        per90[c] = players[c] / players.mins * 90
    per90["pass_pct"] = players.passes_done / players.passes
    per90["goals_total"] = players.goals
    per90["xg_total"] = players.xg
    per90["assists_total"] = players.assists
    per90["league"] = name
    return per90

frames = []
for key, name in leagues.items():
    df = build_league(key, name)
    print(name, df.shape)
    frames.append(df)

per90 = pd.concat(frames)
print("повторяющихся имён:", per90.index.duplicated().sum())
per90.to_pickle(f"{folder}/all_1516_per90.pkl")

print(pd.crosstab(per90.league, per90.role))
top = per90.sort_values("goals_total", ascending=False).groupby("league").head(3)
print(top[["league", "team", "role", "mins", "goals_total", "xg_total"]].sort_values("league"))

In [ ]:
d = per90[per90.index.duplicated(keep=False)]
print(d[["league", "team", "role", "mins", "player_id"]].sort_index())

In [ ]:
per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]
per90.to_pickle(f"{folder}/all_1516_per90_clean.pkl")
print(per90.shape, "дублей:", per90.index.duplicated().sum())

In [ ]:
import os, subprocess
import duckdb

db_path = f"{folder}/transfermarkt-datasets.duckdb"
url = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb"

if not os.path.exists(db_path):
    subprocess.run(["curl", "-L", "-o", db_path, url], check=True)

print("размер файла, МБ:", round(os.path.getsize(db_path) / 1e6))

con = duckdb.connect(db_path, read_only=True)
print(con.sql("SHOW TABLES"))

for t in ["players", "player_valuations"]:
    print("\n==", t)
    print(con.sql(f"DESCRIBE {t}").df()[["column_name", "column_type"]])

print(con.sql("SELECT MIN(date) AS first, MAX(date) AS last, COUNT(*) AS n FROM player_valuations").df())
print(con.sql("SELECT * FROM players LIMIT 3").df().T)

In [ ]:
import re, unicodedata
import duckdb

db_path = f"{folder}/transfermarkt-datasets.duckdb"
con = duckdb.connect(db_path, read_only=True)

val = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club, v.player_club_domestic_competition_id AS tm_comp,
           p.name AS tm_name, p.date_of_birth, p.position AS tm_position
    FROM player_valuations v
    JOIN players p USING (player_id)
    WHERE v.date >= DATE '2015-09-01' AND v.date <= DATE '2016-06-30'
      AND v.player_club_domestic_competition_id IN ('IT1', 'GB1', 'ES1', 'FR1')
""").df()

val["tm_date"] = pd.to_datetime(val["tm_date"])
val = val.sort_values("tm_date").groupby("tm_id").tail(1)      # последняя оценка до конца сезона
val = val[val.value_eur.notna() & (val.value_eur > 0)].copy()
val["age"] = ((pd.Timestamp("2016-06-30") - pd.to_datetime(val.date_of_birth)).dt.days / 365.25).round(1)

print("игроков с оценкой:", len(val))
print(val.groupby("tm_comp").size())
print(val.tm_date.dt.to_period("M").value_counts().sort_index())
print(val.sort_values("value_eur", ascending=False)[["tm_name", "tm_club", "value_eur", "age"]].head(8))

In [ ]:
import difflib

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s.replace("-", " "))
    return " ".join(s.split())

per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]          # дубли: оставляем где больше минут

comp_of = {"Serie A": "IT1", "Premier League": "GB1", "La Liga": "ES1", "Ligue 1": "FR1"}
compatible = {
    "CB": {"Defender"}, "FB": {"Defender"},
    "DM": {"Midfield", "Defender"}, "CM": {"Midfield"},
    "AM_W": {"Midfield", "Attack"}, "ST": {"Attack"},
}

val["norm"] = val.tm_name.map(norm)
val["tokens"] = val.norm.map(lambda s: set(s.split()))

rows = []
for name, r in per90.iterrows():
    n = norm(name)
    toks = set(n.split())
    pool = val[val.tm_comp == comp_of[r.league]]
    how, pick = None, None

    exact = pool[pool.norm == n]
    if len(exact) >= 1:
        how, pick = "exact", exact.iloc[0]
    else:
        sub = pool[pool.tokens.map(lambda t: len(t) > 0 and t <= toks)]   # слова TM-имени есть в полном имени
        if len(sub) > 1:
            sub2 = sub[sub.tm_position.isin(compatible[r.role])]
            sub = sub2 if len(sub2) >= 1 else sub
        if len(sub) == 1:
            how, pick = "tokens", sub.iloc[0]
        elif len(sub) > 1:
            how = "ambiguous"
        else:
            best = difflib.get_close_matches(n, pool.norm.tolist(), n=1, cutoff=0.85)
            if best:
                how, pick = "fuzzy", pool[pool.norm == best[0]].iloc[0]
            else:
                how = "unmatched"
    rows.append((name, how, None if pick is None else pick.tm_id))

mt = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
print(mt.how.value_counts())

merged = per90.join(mt)
merged = merged.merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index(per90.index)
merged.to_pickle(f"{folder}/all_1516_merged.pkl")

print("\nдоля сопоставленных по лигам и ролям:")
print(merged.assign(ok=merged.value_eur.notna()).pivot_table(index="league", columns="role", values="ok", aggfunc="mean").round(2))

print("\nнесопоставленные с наибольшим числом минут:")
um = merged[merged.value_eur.isna()].sort_values("mins", ascending=False)
print(um[["league", "team", "role", "mins", "how"]].head(25))

print("\nсамые дорогие в итоговой таблице:")
print(merged.sort_values("value_eur", ascending=False)[["tm_name", "team", "value_eur", "age"]].head(10))

print("\nпроверка на вид (fuzzy и tokens, первые 15):")
chk = merged[merged.how.isin(["fuzzy", "tokens"])]
print(pd.DataFrame({"statsbomb": chk.index, "transfermarkt": chk.tm_name.values}).head(15))

In [ ]:
print(val.groupby("tm_comp").tm_club.nunique())

In [ ]:
names = ["marek hamsik", "mesut ozil", "glenn whelan", "odion ighalo",
         "radja nainggolan", "cesc fabregas", "vrsaljko", "nacho monreal"]

for n in names:
    q = con.sql(f"""
        SELECT p.player_id, p.name, v.date, v.market_value_in_eur AS value,
               v.current_club_name AS club, v.player_club_domestic_competition_id AS comp
        FROM players p
        LEFT JOIN player_valuations v
          ON v.player_id = p.player_id AND v.date BETWEEN DATE '2015-01-01' AND DATE '2017-01-01'
        WHERE strip_accents(lower(p.name)) LIKE '%{n}%'
        ORDER BY p.player_id, v.date
    """).df()
    print("\n==", n, "| строк:", len(q))
    print(q.to_string(index=False) if len(q) else "игрока нет в таблице players")

In [ ]:
import re, unicodedata, difflib
from collections import Counter
import duckdb

con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

raw = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club,
           p.name AS tm_name, p.date_of_birth, p.position AS tm_position
    FROM player_valuations v
    JOIN players p USING (player_id)
    WHERE v.date >= DATE '2015-07-01' AND v.date <= DATE '2016-09-30'
""").df()
raw["tm_date"] = pd.to_datetime(raw.tm_date)
raw = raw[raw.value_eur.notna() & (raw.value_eur > 0)]

cut = pd.Timestamp("2016-06-30")
before = raw[raw.tm_date <= cut].sort_values("tm_date").groupby("tm_id").tail(1).assign(late=False)
after = (raw[(raw.tm_date > cut) & ~raw.tm_id.isin(before.tm_id)]
         .sort_values("tm_date").groupby("tm_id").head(1).assign(late=True))
val = pd.concat([before, after], ignore_index=True)
val["age"] = ((cut - pd.to_datetime(val.date_of_birth)).dt.days / 365.25).round(1)

print("игроков с оценкой:", len(val), "| из них оценка позже 30 июня 2016:", int(val.late.sum()))
print("клубов:", val.tm_club.nunique())

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s.replace("-", " "))
    return " ".join(s.split())

val["norm"] = val.tm_name.map(norm)
val["tokens"] = val.norm.map(lambda s: frozenset(s.split()))

per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]
per90["norm"] = [norm(n) for n in per90.index]

compatible = {
    "CB": {"Defender"}, "FB": {"Defender"},
    "DM": {"Midfield", "Defender"}, "CM": {"Midfield"},
    "AM_W": {"Midfield", "Attack"}, "ST": {"Attack"},
}

def pick_one(cands, role):
    if len(cands) == 1:
        return cands.iloc[0]
    if len(cands) > 1:
        c2 = cands[cands.tm_position.isin(compatible[role])]
        if len(c2) == 1:
            return c2.iloc[0]
    return None

def find(n, role, pool, min_tokens=1, fuzzy=True):
    toks = frozenset(n.split())
    ex = pool[pool.norm == n]
    if len(ex):
        r = pick_one(ex, role)
        return ("exact", r) if r is not None else ("ambiguous", None)
    sub = pool[pool.tokens.map(lambda t: len(t) >= min_tokens and t <= toks)]
    if len(sub):
        r = pick_one(sub, role)
        return ("tokens", r) if r is not None else ("ambiguous", None)
    if fuzzy:
        best = difflib.get_close_matches(n, pool.norm.tolist(), n=1, cutoff=0.85)
        if best:
            r = pick_one(pool[pool.norm == best[0]], role)
            if r is not None:
                return ("fuzzy", r)
    return ("unmatched", None)

# 1) учим соответствие команда StatsBomb -> клуб Transfermarkt по уверенным совпадениям
votes = {}
for name, r in per90.iterrows():
    how, pick = find(r.norm, r.role, val, min_tokens=2, fuzzy=False)
    if pick is not None and how in ("exact", "tokens"):
        votes.setdefault(r.team, Counter())[pick.tm_club] += 1

team_map = {}
for team, c in votes.items():
    club, n = c.most_common(1)[0]
    if n >= 3 and n / sum(c.values()) >= 0.5:
        team_map[team] = club

# 2) ищем игрока внутри его клуба; если не нашли, то глобально (только надёжные совпадения)
rows = []
for name, r in per90.iterrows():
    club = team_map.get(r.team)
    how, pick = "unmatched", None
    if club is not None:
        how, pick = find(r.norm, r.role, val[val.tm_club == club])
        if pick is not None:
            how = "club_" + how
    if pick is None:
        h2, p2 = find(r.norm, r.role, val, min_tokens=2, fuzzy=False)
        if p2 is not None:
            how, pick = "global_" + h2, p2
    rows.append((name, how, None if pick is None else pick.tm_id))

mt = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
merged = per90.join(mt).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged.to_pickle(f"{folder}/all_1516_merged.pkl")

print(merged.how.value_counts())
print("\nодин tm_id у нескольких игроков:", int(merged.tm_id.dropna().duplicated().sum()))

print("\nдоля сопоставленных по лигам и ролям:")
print(merged.assign(ok=merged.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))

print("\nнесопоставленные с наибольшим числом минут:")
print(merged[merged.value_eur.isna()].sort_values("mins", ascending=False)
      [["league", "team", "role", "mins", "how"]].head(20))

print("\nкоманды без соответствия клуба:", sorted(set(per90.team) - set(team_map)))
print("\nсоответствие команд:")
print(pd.Series(team_map).sort_index().to_string())

print("\nпроверка на вид (global и fuzzy):")
chk = merged[merged.how.str.contains("global|fuzzy", na=False)]
print(pd.DataFrame({"statsbomb": chk.index, "transfermarkt": chk.tm_name.values,
                    "club": chk.tm_club.values}).head(15))

In [ ]:
good_clubs = set(team_map.values())
val_good = val[val.tm_club.isin(good_clubs)]

STOP = {"de", "da", "do", "dos", "das", "del", "la", "le", "van", "von", "di", "el",
        "al", "bin", "junior", "filho"}

def rare(n):
    return {t for t in n.split() if len(t) >= 4 and t not in STOP}

rows = []
for name, r in per90.iterrows():
    pool = val[val.tm_club == team_map[r.team]]
    how, pick = find(r.norm, r.role, pool)
    if pick is not None:
        how = "club_" + how
    else:
        sb = rare(r.norm)
        c = pool[pool.norm.map(lambda s: bool(sb & rare(s))) & pool.tm_position.isin(compatible[r.role])]
        if len(c) == 1:
            how, pick = "club_surname", c.iloc[0]
        else:
            h2, p2 = find(r.norm, r.role, val_good, min_tokens=2, fuzzy=False)
            if p2 is not None:
                how, pick = "global_" + h2, p2
            else:
                how = "unmatched"
    rows.append((name, how, None if pick is None else pick.tm_id))

mt2 = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
merged2 = per90.join(mt2).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged2.to_pickle(f"{folder}/all_1516_merged.pkl")

print(merged2.how.value_counts())

print("\nдоля сопоставленных по лигам и ролям:")
print(merged2.assign(ok=merged2.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))

print("\nодин tm_id у нескольких игроков:")
dup = merged2[merged2.tm_id.notna() & merged2.tm_id.duplicated(keep=False)]
print(dup[["team", "role", "mins", "how", "tm_name"]])

print("\nнужно проверить глазами (surname / global / fuzzy):")
chk = merged2[merged2.how.str.contains("surname|global|fuzzy", na=False)]
print(pd.DataFrame({"statsbomb": chk.index, "team": chk.team.values,
                    "transfermarkt": chk.tm_name.values, "tm_club": chk.tm_club.values,
                    "how": chk.how.values}).to_string())

def candidates(r, k=3):
    pool = val[val.tm_club == team_map[r.team]].copy()
    pool["sim"] = pool.norm.map(lambda s: difflib.SequenceMatcher(None, r.norm, s).ratio())
    pool["pos_ok"] = pool.tm_position.isin(compatible[r.role])
    pool = pool.sort_values(["pos_ok", "sim"], ascending=False).head(k)
    return "; ".join(f"{int(x.tm_id)}:{x.tm_name}[{str(x.tm_position)[:3]}]" for x in pool.itertuples())

print("\nоставшиеся несопоставленные и кандидаты из их клуба:")
rest = merged2[merged2.value_eur.isna()].sort_values("mins", ascending=False)
print(len(rest), "игроков")
for name, r in rest.iterrows():
    print(f"{name} | {r.team} | {r.role} | {int(r.mins)} мин -> {candidates(r)}")

In [ ]:
print(con.sql("DESCRIBE appearances").df()[["column_name", "column_type"]])
print(con.sql("SELECT * FROM appearances LIMIT 2").df().T)

try:
    tm_min = con.sql("""
        SELECT player_id AS tm_id, SUM(minutes_played) AS tm_mins, COUNT(*) AS tm_apps
        FROM appearances
        WHERE date >= DATE '2015-07-01' AND date <= DATE '2016-06-30'
          AND competition_id IN ('IT1', 'GB1', 'ES1', 'FR1')
        GROUP BY player_id
    """).df()
    print("\nигроков с минутами в четырёх лигах:", len(tm_min))

    ok = merged2[merged2.how.isin(["club_exact", "club_tokens"])].reset_index()
    ok = ok.merge(tm_min, on="tm_id", how="left")
    ok["ratio"] = ok.tm_mins / ok.mins
    print("уверенных совпадений без минут в TM:", int(ok.tm_mins.isna().sum()), "из", len(ok))
    print(ok.ratio.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(3))

    bad = ok[(ok.ratio < 0.9) | (ok.ratio > 1.1)]
    print("\nуверенные совпадения с расхождением больше 10%:", len(bad))
    print(bad[["player", "team", "mins", "tm_name", "tm_mins", "how"]].head(15).to_string())
except Exception as ex:
    print("ошибка:", ex)

In [ ]:
STOP = {"de", "da", "do", "dos", "das", "del", "la", "le", "van", "von", "di", "el",
        "al", "bin", "junior", "filho"}

def rare(n):
    return {t for t in n.split() if len(t) >= 4 and t not in STOP}

tm_min = con.sql("""
    SELECT player_id AS tm_id, competition_id AS comp, SUM(minutes_played) AS tm_mins
    FROM appearances
    WHERE date >= DATE '2015-07-01' AND date <= DATE '2016-06-30'
      AND competition_id IN ('IT1', 'GB1', 'ES1', 'FR1')
    GROUP BY player_id, competition_id
""").df()
tm_min = tm_min[tm_min.tm_mins > 0]
mins_lookup = {(int(r.tm_id), r.comp): r.tm_mins for r in tm_min.itertuples()}

comp_of = {"Serie A": "IT1", "Premier League": "GB1", "La Liga": "ES1", "Ligue 1": "FR1"}
LO, HI = 0.88, 1.08

def ratio_of(tm_id, comp, sb_mins):
    m = mins_lookup.get((int(tm_id), comp))
    return None if m is None else m / sb_mins

def ratio_ok(tm_id, comp, sb_mins):
    q = ratio_of(tm_id, comp, sb_mins)
    return q is not None and LO <= q <= HI

def minutes_ok(pool, comp, sb_mins):
    mask = [ratio_ok(i, comp, sb_mins) for i in pool.tm_id]
    return pool[pd.Series(mask, index=pool.index, dtype=bool)]

good_clubs = set(team_map.values())
val_good = val[val.tm_club.isin(good_clubs)]

rows = []
for name, r in per90.iterrows():
    comp = comp_of[r.league]
    n = r.norm
    how, pick = "unmatched", None

    club_pool = minutes_ok(val[val.tm_club == team_map[r.team]], comp, r.mins)
    if len(club_pool):
        h, p = find(n, r.role, club_pool)
        if p is not None:
            how, pick = "club_" + h, p
        else:
            sb = rare(n)
            m = pd.Series([bool(sb & rare(s)) for s in club_pool.norm],
                          index=club_pool.index, dtype=bool)
            c = club_pool[m & club_pool.tm_position.isin(compatible[r.role])]
            if len(c) == 1:
                how, pick = "club_surname", c.iloc[0]
            else:
                c2 = club_pool[club_pool.tm_position.isin(compatible[r.role])]
                if len(c2) == 1:
                    how, pick = "club_minutes", c2.iloc[0]
    if pick is None:
        h, p = find(n, r.role, minutes_ok(val_good, comp, r.mins), min_tokens=2, fuzzy=False)
        if p is not None:
            how, pick = "league_" + h, p
    rows.append((name, how, None if pick is None else pick.tm_id))

mt3 = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
merged3 = per90.join(mt3).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged3["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged3.tm_id, merged3.league, merged3.mins)
]
merged3.to_pickle(f"{folder}/all_1516_merged.pkl")

print(merged3.how.value_counts())
print("\nдоля сопоставленных по лигам и ролям:")
print(merged3.assign(ok=merged3.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))
print("\nодин tm_id у нескольких игроков:", int(merged3.tm_id.dropna().duplicated().sum()))
print("\nотношение минут у сопоставленных:")
print(merged3.min_ratio.astype(float).describe().round(3))

check = ["Fernando Luiz Rosa", "Fernando Francisco Reges", "Jorge Resurrección Merodio",
         "Fábio Henrique Tavares", "Lamine Koné", "Marco Borriello", "Juan Pablo Añor Acosta",
         "Pablo Fornals Malla", "Gabriel Fernández Arenas", "Jorge Luiz Frello Filho",
         "Íñigo Lekue Martínez", "Jesús Joaquín Fernández Sáez de la Torre",
         "Tiago Manuel Dias Correia", "Carlos Henrique Casimiro", "Francesc Fàbregas i Soler"]
print("\nконтрольные игроки:")
print(merged3.loc[[c for c in check if c in merged3.index],
                  ["team", "mins", "tm_name", "tm_club", "how", "min_ratio"]].to_string())

In [ ]:
print("проверить глазами (surname / minutes / league / fuzzy):")
chk = merged3[merged3.how.str.contains("surname|minutes|league|fuzzy", na=False)]
print(pd.DataFrame({"statsbomb": chk.index, "team": chk.team.values,
                    "transfermarkt": chk.tm_name.values, "tm_club": chk.tm_club.values,
                    "how": chk.how.values,
                    "ratio": chk.min_ratio.astype(float).round(2).values}).to_string())

def cands(r, k=3):
    comp = comp_of[r.league]
    pool = val[val.tm_club == team_map[r.team]].copy()
    pool["q"] = pd.Series([ratio_of(i, comp, r.mins) for i in pool.tm_id],
                          index=pool.index, dtype=float)
    pool = pool[pool.q.notna()]
    pool["gap"] = (pool.q - 0.963).abs()
    pool = pool.sort_values("gap").head(k)
    return "; ".join(f"{x.tm_name}[{str(x.tm_position)[:3]}] r={x.q:.2f}" for x in pool.itertuples())

rest = merged3[merged3.value_eur.isna()].sort_values("mins", ascending=False)
print("\nнесопоставленных:", len(rest))
for name, r in rest.iterrows():
    print(f"{name} | {r.team} | {r.role} | {int(r.mins)} мин -> {cands(r)}")

In [ ]:
import re, unicodedata, difflib

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s.replace("-", " "))
    return " ".join(s.split())

raw = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club, p.name AS tm_name,
           p.first_name, p.last_name, p.date_of_birth, p.position AS tm_position
    FROM player_valuations v
    JOIN players p USING (player_id)
    WHERE v.date >= DATE '2015-07-01' AND v.date <= DATE '2016-09-30'
""").df()
raw["tm_date"] = pd.to_datetime(raw.tm_date)
raw = raw[raw.value_eur.notna() & (raw.value_eur > 0)]

cut = pd.Timestamp("2016-06-30")
before = raw[raw.tm_date <= cut].sort_values("tm_date").groupby("tm_id").tail(1).assign(late=False)
after = (raw[(raw.tm_date > cut) & ~raw.tm_id.isin(before.tm_id)]
         .sort_values("tm_date").groupby("tm_id").head(1).assign(late=True))
val = pd.concat([before, after], ignore_index=True)
val["age"] = ((cut - pd.to_datetime(val.date_of_birth)).dt.days / 365.25).round(1)

val["norm"] = val.tm_name.map(norm)
val["norm2"] = (val.first_name.fillna("") + " " + val.last_name.fillna("")).map(norm)
val["tokens"] = val.norm.map(lambda s: frozenset(s.split()))
val["tokens2"] = val.norm2.map(lambda s: frozenset(s.split()))

print(len(val), "игроков с оценкой")
names = ["Koke", "Jorginho", "Casemiro", "Fernandinho", "Suso", "Fabinho", "Juanpi", "Bebé", "Gabi"]
print(val.loc[val.tm_name.isin(names), ["tm_name", "first_name", "last_name", "tm_club"]].to_string())

In [ ]:
per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]
per90["norm"] = [norm(n) for n in per90.index]

good_clubs = set(team_map.values())
val_good = val[val.tm_club.isin(good_clubs)]
PRI = {"club_exact": 0, "league_exact": 0, "club_tokens": 1, "club_surname": 2}

def hits_for(n, pool):
    toks = frozenset(n.split())
    ex = pool[(pool.norm == n) | (pool.norm2 == n)]
    if len(ex):
        return "exact", ex.drop_duplicates("tm_id")
    m1 = pool.tokens.map(lambda t: len(t) > 0 and t <= toks)
    m2 = pool.tokens2.map(lambda t: len(t) > 0 and t <= toks)
    return "tokens", pool[m1 | m2].drop_duplicates("tm_id")

def choose(hit, role, comp, sb_mins):
    if len(hit) == 1:
        return hit.iloc[0]
    if len(hit) > 1:
        h2 = hit[hit.tm_position.isin(compatible[role])]
        if len(h2) == 1:
            return h2.iloc[0]
        base = h2 if len(h2) > 1 else hit
        gaps = sorted((abs(ratio_of(i, comp, sb_mins) - 0.963), k)
                      for k, i in zip(base.index, base.tm_id))
        if len(gaps) > 1 and gaps[1][0] - gaps[0][0] >= 0.02:
            return base.loc[gaps[0][1]]
    return None

rows = []
for name, r in per90.iterrows():
    comp = comp_of[r.league]
    n = r.norm
    how, pick = "unmatched", None

    club_pool = minutes_ok(val[val.tm_club == team_map[r.team]], comp, r.mins)
    if len(club_pool):
        kind, hit = hits_for(n, club_pool)
        pick = choose(hit, r.role, comp, r.mins)
        if pick is not None:
            how = "club_" + kind
        else:
            sb = rare(n)
            m = pd.Series([bool(sb & (rare(a) | rare(b))) for a, b in zip(club_pool.norm, club_pool.norm2)],
                          index=club_pool.index, dtype=bool)
            c = club_pool[m & club_pool.tm_position.isin(compatible[r.role])]
            ok = pd.Series([abs(ratio_of(i, comp, r.mins) - 0.963) <= 0.03 for i in c.tm_id],
                           index=c.index, dtype=bool)
            c = c[ok]
            if len(c) == 1:
                how, pick = "club_surname", c.iloc[0]
    if pick is None:
        lp = minutes_ok(val_good, comp, r.mins)
        ex = lp[(lp.norm == n) | (lp.norm2 == n)].drop_duplicates("tm_id")
        if len(ex) == 1:
            how, pick = "league_exact", ex.iloc[0]
    rows.append((name, how, None if pick is None else pick.tm_id))

mt4 = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
mt4["tm_id"] = pd.to_numeric(mt4.tm_id)
mt4["pri"] = mt4.how.map(lambda h: PRI.get(h, 9))
mt4["gap"] = [
    abs(ratio_of(t, comp_of[per90.loc[p, "league"]], per90.loc[p, "mins"]) - 0.963) if pd.notna(t) else np.nan
    for p, t in zip(mt4.index, mt4.tm_id)
]

dups = mt4[mt4.tm_id.notna() & mt4.tm_id.duplicated(keep=False)].sort_values(["tm_id", "pri", "gap"])
drop = dups.index[dups.duplicated("tm_id", keep="first")]
print("игроков в дублях:", len(dups), "-> пара снята у", len(drop))
mt4.loc[drop, "how"] = "unmatched"
mt4.loc[drop, "tm_id"] = np.nan

merged4 = per90.join(mt4[["how", "tm_id"]]).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged4["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged4.tm_id, merged4.league, merged4.mins)
]
merged4.to_pickle(f"{folder}/all_1516_merged.pkl")
print("сохранено:", merged4.shape)

In [ ]:
print(merged4.how.value_counts())
print("\nдоля сопоставленных по лигам и ролям:")
print(merged4.assign(ok=merged4.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))
print("\nдубли tm_id:", int(merged4.tm_id.dropna().duplicated().sum()))
print(merged4.min_ratio.astype(float).describe().round(3))

check = ["Fernando Luiz Rosa", "Fernando Francisco Reges", "Jorge Resurrección Merodio",
         "Fábio Henrique Tavares", "Lamine Koné", "Marco Borriello", "Juan Pablo Añor Acosta",
         "Pablo Fornals Malla", "Gabriel Fernández Arenas", "Jorge Luiz Frello Filho",
         "Íñigo Lekue Martínez", "Jesús Joaquín Fernández Sáez de la Torre",
         "Tiago Manuel Dias Correia", "Carlos Henrique Casimiro", "Francesc Fàbregas i Soler"]
print("\nконтрольные игроки:")
print(merged4.loc[[c for c in check if c in merged4.index],
                  ["team", "mins", "tm_name", "tm_club", "how", "min_ratio"]].to_string())

print("\nпроверить глазами (surname / league):")
chk = merged4[merged4.how.isin(["club_surname", "league_exact"])]
print(pd.DataFrame({"statsbomb": chk.index, "team": chk.team.values,
                    "transfermarkt": chk.tm_name.values, "tm_club": chk.tm_club.values,
                    "how": chk.how.values,
                    "ratio": chk.min_ratio.astype(float).round(2).values}).to_string())

def cands(r, k=3):
    comp = comp_of[r.league]
    pool = val[val.tm_club == team_map[r.team]].copy()
    pool["q"] = pd.Series([ratio_of(i, comp, r.mins) for i in pool.tm_id], index=pool.index, dtype=float)
    pool = pool[pool.q.notna()]
    pool["gap"] = (pool.q - 0.963).abs()
    pool = pool.sort_values("gap").head(k)
    return "; ".join(f"{x.tm_name}[{str(x.tm_position)[:3]}] r={x.q:.2f}" for x in pool.itertuples())

rest = merged4[merged4.value_eur.isna()].sort_values("mins", ascending=False)
print("\nнесопоставленных:", len(rest))
for name, r in rest.iterrows():
    print(f"{name} | {r.team} | {r.role} | {int(r.mins)} мин -> {cands(r)}")

In [ ]:
import re, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import duckdb
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s.replace("-", " "))
    return " ".join(s.split())

STOP = {"de", "da", "do", "dos", "das", "del", "la", "le", "van", "von", "di", "el",
        "al", "bin", "junior", "filho"}

def rare(n):
    return {t for t in n.split() if len(t) >= 4 and t not in STOP}

compatible = {
    "CB": {"Defender"}, "FB": {"Defender"},
    "DM": {"Midfield", "Defender"}, "CM": {"Midfield"},
    "AM_W": {"Midfield", "Attack"}, "ST": {"Attack"},
}
comp_of = {"Serie A": "IT1", "Premier League": "GB1", "La Liga": "ES1", "Ligue 1": "FR1"}

raw = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club, p.name AS tm_name,
           p.first_name, p.last_name, p.date_of_birth, p.position AS tm_position
    FROM player_valuations v
    JOIN players p USING (player_id)
    WHERE v.date >= DATE '2015-07-01' AND v.date <= DATE '2016-09-30'
""").df()
raw["tm_date"] = pd.to_datetime(raw.tm_date)
raw = raw[raw.value_eur.notna() & (raw.value_eur > 0)]

cut = pd.Timestamp("2016-06-30")
before = raw[raw.tm_date <= cut].sort_values("tm_date").groupby("tm_id").tail(1).assign(late=False)
after = (raw[(raw.tm_date > cut) & ~raw.tm_id.isin(before.tm_id)]
         .sort_values("tm_date").groupby("tm_id").head(1).assign(late=True))
val = pd.concat([before, after], ignore_index=True)
val["age"] = ((cut - pd.to_datetime(val.date_of_birth)).dt.days / 365.25).round(1)

val["norm"] = val.tm_name.map(norm)
val["norm2"] = (val.first_name.fillna("") + " " + val.last_name.fillna("")).map(norm)
val["tokens"] = val.norm.map(lambda s: frozenset(s.split()))
val["tokens2"] = val.norm2.map(lambda s: frozenset(s.split()))

print(len(val), "игроков с оценкой")
nick = ["Koke", "Jorginho", "Casemiro", "Fernandinho", "Suso", "Fabinho", "Juanpi", "Bebé", "Gabi"]
print(val.loc[val.tm_name.isin(nick), ["tm_name", "first_name", "last_name", "tm_club"]].to_string())

In [ ]:
per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]
per90["norm"] = [norm(n) for n in per90.index]

def hits_for(n, pool, min_tokens=1):
    toks = frozenset(n.split())
    ex = pool[(pool.norm == n) | (pool.norm2 == n)]
    if len(ex):
        return "exact", ex.drop_duplicates("tm_id")
    m1 = pool.tokens.map(lambda t: len(t) >= min_tokens and t <= toks)
    m2 = pool.tokens2.map(lambda t: len(t) >= min_tokens and t <= toks)
    return "tokens", pool[m1 | m2].drop_duplicates("tm_id")

votes = {}
for name, r in per90.iterrows():
    kind, hit = hits_for(r.norm, val, min_tokens=2)
    if len(hit) > 1:
        hit = hit[hit.tm_position.isin(compatible[r.role])]
    if len(hit) == 1:
        votes.setdefault(r.team, Counter())[hit.iloc[0].tm_club] += 1

team_map = {}
for team, c in votes.items():
    club, k = c.most_common(1)[0]
    if k >= 3 and k / sum(c.values()) >= 0.5:
        team_map[team] = club

print("команд сопоставлено:", len(team_map), "из", per90.team.nunique())
print("без пары:", sorted(set(per90.team) - set(team_map)))
for t in ["Barcelona", "Lyon", "Atlético Madrid", "Inter Milan", "Napoli", "Rennes", "Lazio"]:
    print(t, "->", team_map.get(t))

In [ ]:
tm_min = con.sql("""
    SELECT player_id AS tm_id, competition_id AS comp, SUM(minutes_played) AS tm_mins
    FROM appearances
    WHERE date >= DATE '2015-07-01' AND date <= DATE '2016-06-30'
      AND competition_id IN ('IT1', 'GB1', 'ES1', 'FR1')
    GROUP BY player_id, competition_id
""").df()
tm_min = tm_min[tm_min.tm_mins > 0]
mins_lookup = {(int(r.tm_id), r.comp): r.tm_mins for r in tm_min.itertuples()}
LO, HI = 0.88, 1.08

def ratio_of(tm_id, comp, sb_mins):
    m = mins_lookup.get((int(tm_id), comp))
    return None if m is None else m / sb_mins

def ratio_ok(tm_id, comp, sb_mins):
    q = ratio_of(tm_id, comp, sb_mins)
    return q is not None and LO <= q <= HI

def minutes_ok(pool, comp, sb_mins):
    mask = [ratio_ok(i, comp, sb_mins) for i in pool.tm_id]
    return pool[pd.Series(mask, index=pool.index, dtype=bool)]

def choose(hit, role, comp, sb_mins):
    if len(hit) == 1:
        return hit.iloc[0]
    if len(hit) > 1:
        h2 = hit[hit.tm_position.isin(compatible[role])]
        if len(h2) == 1:
            return h2.iloc[0]
        base = h2 if len(h2) > 1 else hit
        gaps = sorted((abs(ratio_of(i, comp, sb_mins) - 0.963), k)
                      for k, i in zip(base.index, base.tm_id))
        if len(gaps) > 1 and gaps[1][0] - gaps[0][0] >= 0.02:
            return base.loc[gaps[0][1]]
    return None

good_clubs = set(team_map.values())
val_good = val[val.tm_club.isin(good_clubs)]
PRI = {"club_exact": 0, "league_exact": 0, "club_tokens": 1, "club_surname": 2}

rows = []
for name, r in per90.iterrows():
    comp = comp_of[r.league]
    n = r.norm
    how, pick = "unmatched", None
    club = team_map.get(r.team)
    club_pool = minutes_ok(val[val.tm_club == club], comp, r.mins) if club else val.iloc[0:0]

    if len(club_pool):
        kind, hit = hits_for(n, club_pool)
        pick = choose(hit, r.role, comp, r.mins)
        if pick is not None:
            how = "club_" + kind
        else:
            sb = rare(n)
            m = pd.Series([bool(sb & (rare(a) | rare(b))) for a, b in zip(club_pool.norm, club_pool.norm2)],
                          index=club_pool.index, dtype=bool)
            c = club_pool[m & club_pool.tm_position.isin(compatible[r.role])]
            ok = pd.Series([abs(ratio_of(i, comp, r.mins) - 0.963) <= 0.03 for i in c.tm_id],
                           index=c.index, dtype=bool)
            c = c[ok]
            if len(c) == 1:
                how, pick = "club_surname", c.iloc[0]
    if pick is None:
        lp = minutes_ok(val_good, comp, r.mins)
        ex = lp[(lp.norm == n) | (lp.norm2 == n)].drop_duplicates("tm_id")
        if len(ex) == 1:
            how, pick = "league_exact", ex.iloc[0]
    rows.append((name, how, None if pick is None else pick.tm_id))

mt4 = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
mt4["tm_id"] = pd.to_numeric(mt4.tm_id)
mt4["pri"] = mt4.how.map(lambda h: PRI.get(h, 9))
mt4["gap"] = [
    abs(ratio_of(t, comp_of[per90.loc[p, "league"]], per90.loc[p, "mins"]) - 0.963)
    if pd.notna(t) else np.nan
    for p, t in zip(mt4.index, mt4.tm_id)
]

dups = mt4[mt4.tm_id.notna() & mt4.tm_id.duplicated(keep=False)].sort_values(["tm_id", "pri", "gap"])
drop = dups.index[dups.duplicated("tm_id", keep="first")]
print("игроков в дублях:", len(dups), "-> пара снята у", len(drop))
mt4.loc[drop, "how"] = "unmatched"
mt4.loc[drop, "tm_id"] = np.nan

merged4 = per90.join(mt4[["how", "tm_id"]]).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged4["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged4.tm_id, merged4.league, merged4.mins)
]
merged4.to_pickle(f"{folder}/all_1516_merged.pkl")
print("сохранено:", merged4.shape)

In [ ]:
print(merged4.how.value_counts())
print("\nдоля сопоставленных по лигам и ролям:")
print(merged4.assign(ok=merged4.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))
print("\nдубли tm_id:", int(merged4.tm_id.dropna().duplicated().sum()))
print(merged4.min_ratio.astype(float).describe().round(3))

check = ["Fernando Luiz Rosa", "Fernando Francisco Reges", "Jorge Resurrección Merodio",
         "Fábio Henrique Tavares", "Lamine Koné", "Marco Borriello", "Juan Pablo Añor Acosta",
         "Pablo Fornals Malla", "Gabriel Fernández Arenas", "Jorge Luiz Frello Filho",
         "Íñigo Lekue Martínez", "Jesús Joaquín Fernández Sáez de la Torre",
         "Tiago Manuel Dias Correia", "Carlos Henrique Casimiro", "Francesc Fàbregas i Soler"]
print("\nконтрольные игроки:")
print(merged4.loc[[c for c in check if c in merged4.index],
                  ["team", "mins", "tm_name", "tm_club", "how", "min_ratio"]].to_string())

print("\nпроверить глазами (surname / league):")
chk = merged4[merged4.how.isin(["club_surname", "league_exact"])]
print(pd.DataFrame({"statsbomb": chk.index, "team": chk.team.values,
                    "transfermarkt": chk.tm_name.values, "tm_club": chk.tm_club.values,
                    "how": chk.how.values,
                    "ratio": chk.min_ratio.astype(float).round(2).values}).to_string())

def cands(r, k=3):
    comp = comp_of[r.league]
    club = team_map.get(r.team)
    pool = val[val.tm_club == club].copy() if club else val.iloc[0:0].copy()
    pool["q"] = pd.Series([ratio_of(i, comp, r.mins) for i in pool.tm_id], index=pool.index, dtype=float)
    pool = pool[pool.q.notna()]
    pool["gap"] = (pool.q - 0.963).abs()
    pool = pool.sort_values("gap").head(k)
    return "; ".join(f"{x.tm_name}[{str(x.tm_position)[:3]}] r={x.q:.2f}" for x in pool.itertuples())

rest = merged4[merged4.value_eur.isna()].sort_values("mins", ascending=False)
print("\nнесопоставленных:", len(rest))
for name, r in rest.iterrows():
    print(f"{name} | {r.team} | {r.role} | {int(r.mins)} мин -> {cands(r)}")

In [ ]:
import re, unicodedata, difflib

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z\s]", " ", s.replace("-", " "))
    return " ".join(s.split())

raw = con.sql("""
    SELECT v.player_id AS tm_id, v.date AS tm_date, v.market_value_in_eur AS value_eur,
           v.current_club_name AS tm_club, p.name AS tm_name,
           p.first_name, p.last_name, p.date_of_birth, p.position AS tm_position
    FROM player_valuations v
    JOIN players p USING (player_id)
    WHERE v.date >= DATE '2015-07-01' AND v.date <= DATE '2016-09-30'
""").df()
raw["tm_date"] = pd.to_datetime(raw.tm_date)
raw = raw[raw.value_eur.notna() & (raw.value_eur > 0)]

cut = pd.Timestamp("2016-06-30")
before = raw[raw.tm_date <= cut].sort_values("tm_date").groupby("tm_id").tail(1).assign(late=False)
after = (raw[(raw.tm_date > cut) & ~raw.tm_id.isin(before.tm_id)]
         .sort_values("tm_date").groupby("tm_id").head(1).assign(late=True))
val = pd.concat([before, after], ignore_index=True)
val["age"] = ((cut - pd.to_datetime(val.date_of_birth)).dt.days / 365.25).round(1)

val["norm"] = val.tm_name.map(norm)
val["norm2"] = (val.first_name.fillna("") + " " + val.last_name.fillna("")).map(norm)
val["tokens"] = val.norm.map(lambda s: frozenset(s.split()))
val["tokens2"] = val.norm2.map(lambda s: frozenset(s.split()))

print(len(val), "игроков с оценкой")
names = ["Koke", "Jorginho", "Casemiro", "Fernandinho", "Suso", "Fabinho", "Juanpi", "Bebé", "Gabi"]
print(val.loc[val.tm_name.isin(names), ["tm_name", "first_name", "last_name", "tm_club"]].to_string())

In [ ]:
per90 = pd.read_pickle(f"{folder}/all_1516_per90.pkl")
per90 = per90.sort_values("mins", ascending=False)
per90 = per90[~per90.index.duplicated(keep="first")]
per90["norm"] = [norm(n) for n in per90.index]

good_clubs = set(team_map.values())
val_good = val[val.tm_club.isin(good_clubs)]
PRI = {"club_exact": 0, "league_exact": 0, "club_tokens": 1, "club_surname": 2}

def hits_for(n, pool):
    toks = frozenset(n.split())
    ex = pool[(pool.norm == n) | (pool.norm2 == n)]
    if len(ex):
        return "exact", ex.drop_duplicates("tm_id")
    m1 = pool.tokens.map(lambda t: len(t) > 0 and t <= toks)
    m2 = pool.tokens2.map(lambda t: len(t) > 0 and t <= toks)
    return "tokens", pool[m1 | m2].drop_duplicates("tm_id")

def choose(hit, role, comp, sb_mins):
    if len(hit) == 1:
        return hit.iloc[0]
    if len(hit) > 1:
        h2 = hit[hit.tm_position.isin(compatible[role])]
        if len(h2) == 1:
            return h2.iloc[0]
        base = h2 if len(h2) > 1 else hit
        gaps = sorted((abs(ratio_of(i, comp, sb_mins) - 0.963), k)
                      for k, i in zip(base.index, base.tm_id))
        if len(gaps) > 1 and gaps[1][0] - gaps[0][0] >= 0.02:
            return base.loc[gaps[0][1]]
    return None

rows = []
for name, r in per90.iterrows():
    comp = comp_of[r.league]
    n = r.norm
    how, pick = "unmatched", None

    club_pool = minutes_ok(val[val.tm_club == team_map[r.team]], comp, r.mins)
    if len(club_pool):
        kind, hit = hits_for(n, club_pool)
        pick = choose(hit, r.role, comp, r.mins)
        if pick is not None:
            how = "club_" + kind
        else:
            sb = rare(n)
            m = pd.Series([bool(sb & (rare(a) | rare(b))) for a, b in zip(club_pool.norm, club_pool.norm2)],
                          index=club_pool.index, dtype=bool)
            c = club_pool[m & club_pool.tm_position.isin(compatible[r.role])]
            ok = pd.Series([abs(ratio_of(i, comp, r.mins) - 0.963) <= 0.03 for i in c.tm_id],
                           index=c.index, dtype=bool)
            c = c[ok]
            if len(c) == 1:
                how, pick = "club_surname", c.iloc[0]
    if pick is None:
        lp = minutes_ok(val_good, comp, r.mins)
        ex = lp[(lp.norm == n) | (lp.norm2 == n)].drop_duplicates("tm_id")
        if len(ex) == 1:
            how, pick = "league_exact", ex.iloc[0]
    rows.append((name, how, None if pick is None else pick.tm_id))

mt4 = pd.DataFrame(rows, columns=["player", "how", "tm_id"]).set_index("player")
mt4["tm_id"] = pd.to_numeric(mt4.tm_id)
mt4["pri"] = mt4.how.map(lambda h: PRI.get(h, 9))
mt4["gap"] = [
    abs(ratio_of(t, comp_of[per90.loc[p, "league"]], per90.loc[p, "mins"]) - 0.963) if pd.notna(t) else np.nan
    for p, t in zip(mt4.index, mt4.tm_id)
]

dups = mt4[mt4.tm_id.notna() & mt4.tm_id.duplicated(keep=False)].sort_values(["tm_id", "pri", "gap"])
drop = dups.index[dups.duplicated("tm_id", keep="first")]
print("игроков в дублях:", len(dups), "-> пара снята у", len(drop))
mt4.loc[drop, "how"] = "unmatched"
mt4.loc[drop, "tm_id"] = np.nan

merged4 = per90.join(mt4[["how", "tm_id"]]).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged4["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged4.tm_id, merged4.league, merged4.mins)
]
merged4.to_pickle(f"{folder}/all_1516_merged.pkl")
print("сохранено:", merged4.shape)

In [ ]:
print(merged4.how.value_counts())
print("\nдоля сопоставленных по лигам и ролям:")
print(merged4.assign(ok=merged4.value_eur.notna()).pivot_table(
    index="league", columns="role", values="ok", aggfunc="mean").round(2))
print("\nдубли tm_id:", int(merged4.tm_id.dropna().duplicated().sum()))
print(merged4.min_ratio.astype(float).describe().round(3))

check = ["Fernando Luiz Rosa", "Fernando Francisco Reges", "Jorge Resurrección Merodio",
         "Fábio Henrique Tavares", "Lamine Koné", "Marco Borriello", "Juan Pablo Añor Acosta",
         "Pablo Fornals Malla", "Gabriel Fernández Arenas", "Jorge Luiz Frello Filho",
         "Íñigo Lekue Martínez", "Jesús Joaquín Fernández Sáez de la Torre",
         "Tiago Manuel Dias Correia", "Carlos Henrique Casimiro", "Francesc Fàbregas i Soler"]
print("\nконтрольные игроки:")
print(merged4.loc[[c for c in check if c in merged4.index],
                  ["team", "mins", "tm_name", "tm_club", "how", "min_ratio"]].to_string())

print("\nпроверить глазами (surname / league):")
chk = merged4[merged4.how.isin(["club_surname", "league_exact"])]
print(pd.DataFrame({"statsbomb": chk.index, "team": chk.team.values,
                    "transfermarkt": chk.tm_name.values, "tm_club": chk.tm_club.values,
                    "how": chk.how.values,
                    "ratio": chk.min_ratio.astype(float).round(2).values}).to_string())

def cands(r, k=3):
    comp = comp_of[r.league]
    pool = val[val.tm_club == team_map[r.team]].copy()
    pool["q"] = pd.Series([ratio_of(i, comp, r.mins) for i in pool.tm_id], index=pool.index, dtype=float)
    pool = pool[pool.q.notna()]
    pool["gap"] = (pool.q - 0.963).abs()
    pool = pool.sort_values("gap").head(k)
    return "; ".join(f"{x.tm_name}[{str(x.tm_position)[:3]}] r={x.q:.2f}" for x in pool.itertuples())

rest = merged4[merged4.value_eur.isna()].sort_values("mins", ascending=False)
print("\nнесопоставленных:", len(rest))
for name, r in rest.iterrows():
    print(f"{name} | {r.team} | {r.role} | {int(r.mins)} мин -> {cands(r)}")

In [ ]:
manual = {
    "Fábio Henrique Tavares": ("Fabinho", "AS Monaco"),
    "Robert Brady": ("Robbie Brady", "Norwich City"),
    "Jorge Resurrección Merodio": ("Koke", "Atlético de Madrid"),
    "Jorge Luiz Frello Filho": ("Jorginho", "SSC Napoli"),
    "Gabriel Fernández Arenas": ("Gabi", "Atlético de Madrid"),
    "Jonathan Rodríguez Menéndez": ("Jony", "Sporting Gijón"),
    "José Luis García del Pozo": ("Recio", "Málaga CF"),
    "Fernando Luiz Rosa": ("Fernandinho", "Manchester City"),
    "Roberto Román Triguero": ("Tito", "Rayo Vallecano"),
    "Manuel Agudo Durán": ("Nolito", "Celta de Vigo"),
    "Sergio Gontán Gallardo": ("Keko Gontán", "SD Eibar"),
    "Damjan Đoković": ("Damjan Djokovic", "GFC Ajaccio"),
    "Tiago Manuel Dias Correia": ("Bebé", "Rayo Vallecano"),
    "Augusto Matías Fernández": ("Augusto Fernández", "Atlético de Madrid"),
    "Wergiton do Rosário Calmon": ("Somália", "FC Toulouse"),
    "Pedro Taunausú Domínguez Placeres": ("Tana", "UD Las Palmas"),
    "Víctor Machín Pérez": ("Vitolo", "Sevilla FC"),
    "Joaquín José Marín Ruiz": ("Quini", "Rayo Vallecano"),
    "Emil Hallfreðsson": ("Emil Hallfredsson", "Hellas Verona"),
    "Kléper Laveran Lima Ferreira": ("Pepe", "Real Madrid"),
    "Marcos Aoás Corrêa": ("Marquinhos", "Paris Saint-Germain"),
    "Gabriel Armando de Abreu": ("Gabriel Paulista", "Arsenal FC"),
    "Francisco Román Alarcón Suárez": ("Isco", "Real Madrid"),
    "Jorge Andújar Moreno": ("Coke", "Sevilla FC"),
    "Samuel Castillejo Azuaga": ("Samu Castillejo", "Villarreal CF"),
    "Juan Pablo Añor Acosta": ("Juanpi", "Málaga CF"),
    "Gnaly Maxwell Cornet": ("Maxwel Cornet", "Olympique Lyon"),
    "Laureano Sanabria Ruiz": ("Laure", "Deportivo de La Coruña"),
    "Carlos Henrique Casimiro": ("Casemiro", "Real Madrid"),
    "Juan Francisco García García": ("Juanfran", "Levante UD"),
    "Armindo Tué Na Bangna": ("Bruma", "Real Sociedad"),
    "Gonzalo Castro Irizábal": ("Chory Castro", "Málaga CF"),
    "Jerónimo Figueroa Cabrera": ("Momo", "UD Las Palmas"),
    "Edson André Sitoe": ("Mexer", "Stade Rennais FC"),
    "Juan Carlos Pérez López": ("Juankar", "Málaga CF"),
    "Charly Musonda Junior": ("Charly Musonda Jr.", "Real Betis Balompié"),
    "Jesús Joaquín Fernández Sáez de la Torre": ("Suso", "Genoa CFC"),
    "Íñigo Lekue Martínez": ("Iñigo Lekue", "Athletic Bilbao"),
    "Maximiliano Gastón López": ("Maxi López", "Torino FC"),
    "Luis Carlos Correia Pinto": ("Luísinho", "Deportivo de La Coruña"),
    "Andrew Philip King": ("Andy King", "Leicester City"),
    "José Luis Sanmartín Mato": ("Joselu", "Stoke City"),
    "Kevin Mirallas": ("Kevin Mirallas", "Everton FC"),
    "Sérgio Paulo Barbosa Valente": ("Duda", "Málaga CF"),
    "Éderzito António Macedo Lopes": ("Éder", "LOSC Lille"),
    "Alejandro Menéndez Díez": ("Álex Menéndez", "Sporting Gijón"),
    "Alberto Mateus Contreiras Gonçalves": ("Manucho", "Rayo Vallecano"),
    "Moisés Gómez Bordonado": ("Moi Gómez", "Getafe CF"),
    "Rubén Gracia Calmache": ("Cani", "Deportivo de La Coruña"),
    "Eder Citadin Martins": ("Éder", "Inter Milan"),
    "José Rodolfo Pires Ribeiro": ("Dodô", "UC Sampdoria"),
    "Sebastián Coates Nión": ("Sebastián Coates", "Sunderland AFC"),
}

LO_M, HI_M = 0.85, 1.10

def apply_manual(mt, manual):
    done, failed = 0, []
    for sb_name, (tm_name, tm_club) in manual.items():
        if sb_name not in per90.index:
            failed.append((sb_name, "нет в StatsBomb"))
            continue
        c = val[(val.norm == norm(tm_name)) & (val.tm_club == tm_club)].drop_duplicates("tm_id")
        if len(c) != 1:
            failed.append((sb_name, f"кандидатов: {len(c)}"))
            continue
        tid = c.iloc[0].tm_id
        r = per90.loc[sb_name]
        q = ratio_of(tid, comp_of[r.league], r.mins)
        if q is None or not (LO_M <= q <= HI_M):
            failed.append((sb_name, f"минуты не сходятся: {q}"))
            continue
        other = mt.tm_id == tid
        mt.loc[other, "how"] = "unmatched"
        mt.loc[other, "tm_id"] = np.nan
        mt.loc[sb_name, "how"] = "manual"
        mt.loc[sb_name, "tm_id"] = tid
        done += 1
    print("добавлено вручную:", done)
    for f in failed:
        print("не принято:", f)
    return mt

mt4 = apply_manual(mt4, manual)

merged4 = per90.join(mt4[["how", "tm_id"]]).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged4["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged4.tm_id, merged4.league, merged4.mins)
]
merged4.to_pickle(f"{folder}/all_1516_merged.pkl")
print("сохранено:", merged4.shape, "| без оценки:", int(merged4.value_eur.isna().sum()))
print("дубли tm_id:", int(merged4.tm_id.dropna().duplicated().sum()))
print(merged4.how.value_counts())

print("\nвручную добавленные:")
mm = merged4[merged4.how == "manual"]
print(pd.DataFrame({"statsbomb": mm.index, "team": mm.team.values, "transfermarkt": mm.tm_name.values,
                    "tm_club": mm.tm_club.values, "ratio": mm.min_ratio.astype(float).round(2).values}).to_string())

In [ ]:
extra = {
    "Emil Hallfreðsson": "Emil Hallfreðsson",
    "Sebastián Coates Nión": "Sebastián Coates",
    "José Rodolfo Pires Ribeiro": "Dodô",
    "Íñigo Lekue Martínez": "Iñigo Lekue",
}

for sb_name, tm_name in extra.items():
    if sb_name not in per90.index:
        print("нет в StatsBomb:", sb_name)
        continue
    r = per90.loc[sb_name]
    c = val[val.norm == norm(tm_name)].drop_duplicates("tm_id")
    ok = []
    for i in c.tm_id:
        q = ratio_of(i, comp_of[r.league], r.mins)
        if q is not None and LO_M <= q <= HI_M:
            ok.append(i)
    if len(ok) != 1:
        print("не принято:", sb_name, "| кандидатов с подходящими минутами:", len(ok))
        continue
    tid = ok[0]
    other = mt4.tm_id == tid
    mt4.loc[other, "how"] = "unmatched"
    mt4.loc[other, "tm_id"] = np.nan
    mt4.loc[sb_name, "how"] = "manual"
    mt4.loc[sb_name, "tm_id"] = tid
    print("ok:", sb_name, "->", c[c.tm_id == tid].iloc[0].tm_name, "|",
          c[c.tm_id == tid].iloc[0].tm_club, "| ratio", round(ratio_of(tid, comp_of[r.league], r.mins), 2))

merged4 = per90.join(mt4[["how", "tm_id"]]).reset_index().merge(
    val[["tm_id", "tm_name", "tm_club", "tm_date", "late", "value_eur", "age", "tm_position"]],
    on="tm_id", how="left",
).set_index("player")
merged4["min_ratio"] = [
    np.nan if pd.isna(t) else ratio_of(t, comp_of[l], m)
    for t, l, m in zip(merged4.tm_id, merged4.league, merged4.mins)
]
merged4.to_pickle(f"{folder}/all_1516_merged.pkl")

print("\nсохранено:", merged4.shape, "| без оценки:", int(merged4.value_eur.isna().sum()),
      "| с флагом late:", int(merged4.late.fillna(False).sum()))
print("дубли tm_id:", int(merged4.tm_id.dropna().duplicated().sum()))

print("\nостались без оценки:")
rest = merged4[merged4.value_eur.isna()].sort_values("mins", ascending=False)
print(rest[["league", "team", "role", "mins"]].to_string())

In [ ]:
import statsmodels.formula.api as smf

spec = {
    "ST":   {"xg": 1, "shots": 1, "key_passes": 1, "dribbles_won": 1,
             "prog_carries": 1, "pressures": 1, "fouls_won": 1},
    "AM_W": {"xg": 1, "key_passes": 1, "dribbles_won": 1, "prog_carries": 1,
             "prog_passes": 1, "shots": 1, "pressures": 1, "recoveries": 1},
    "CM":   {"prog_passes": 1, "key_passes": 1, "final_third_passes": 1, "pass_pct": 1,
             "prog_carries": 1, "tackles": 1, "interceptions": 1, "recoveries": 1, "pressures": 1},
    "DM":   {"prog_passes": 1, "pass_pct": 1, "long_passes": 1, "tackles": 1,
             "interceptions": 1, "recoveries": 1, "pressures": 1, "dispossessed": -1},
    "FB":   {"prog_carries": 1, "prog_passes": 1, "crosses": 1, "key_passes": 1,
             "dribbles_won": 1, "tackles": 1, "interceptions": 1, "recoveries": 1},
    "CB":   {"prog_passes": 1, "long_passes": 1, "pass_pct": 1, "interceptions": 1,
             "tackles": 1, "clearances": 1, "blocks": 1, "recoveries": 1,
             "aerial_lost": -1, "fouls": -1},
}

d = merged4.copy()
d["score"] = np.nan
for role, metrics in spec.items():
    idx = d.role == role
    parts = []
    for m, w in metrics.items():
        pct = d.loc[idx, m].rank(pct=True)
        parts.append(pct if w > 0 else 1 - pct)
    d.loc[idx, "score"] = pd.concat(parts, axis=1).mean(axis=1) * 100

print("лучшие по оценке игры в каждой роли (проверка здравого смысла):")
for role in spec:
    t = d[d.role == role].sort_values("score", ascending=False).head(5)
    print(f"\n{role}:", "; ".join(f"{n} ({r.team}, {r.score:.0f})" for n, r in t.iterrows()))

d2 = d[d.value_eur.notna()].copy()
d2["log_value"] = np.log(d2.value_eur)
d2["value_m"] = d2.value_eur / 1e6

print("\nмодели по ролям:")
out = []
for role in spec:
    s = d2[d2.role == role].copy()
    mod = smf.ols("log_value ~ score + age + I(age**2) + C(league)", data=s).fit()
    s["expected_m"] = np.exp(mod.fittedvalues) / 1e6
    s["resid"] = mod.resid
    print(f"{role}: n={len(s)}, R²={mod.rsquared:.2f}, эффект +10 пунктов оценки = "
          f"{(np.exp(mod.params['score'] * 10) - 1) * 100:+.0f}% к стоимости")
    out.append(s)
res = pd.concat(out)
res.to_pickle(f"{folder}/all_1516_scored.pkl")

print("\nкандидаты «недооценённые» (1500+ минут), по ролям:")
for role in spec:
    s = res[(res.role == role) & (res.mins >= 1500)].sort_values("resid").head(8)
    print(f"\n{role}")
    print(pd.DataFrame({
        "team": s.team, "age": s.age, "score": s.score.round(0),
        "value_m": s.value_m.round(1), "expected_m": s.expected_m.round(1),
        "late": s.late,
    }).to_string())

In [ ]:
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/football_portfolio"
res = pd.read_pickle(f"{folder}/all_1516_scored.pkl")
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
res["score_pct"] = res.groupby("role").score.rank(pct=True)

print("1) где в своей роли стоят известные игроки (место / из скольких, оценка):")
stars = ["Lionel Andrés Messi Cuccittini", "Cristiano Ronaldo dos Santos Aveiro",
         "Neymar da Silva Santos Junior", "Luis Alberto Suárez Díaz", "Gonzalo Gerardo Higuaín",
         "Zlatan Ibrahimović", "Sergio Leonel Agüero del Castillo", "Harry Kane", "Jamie Vardy",
         "Antoine Griezmann", "Paul Pogba", "Luka Modrić", "Marco Verratti"]
for n in stars:
    if n in res.index:
        r = res.loc[n]
        sub = res[res.role == r.role]
        place = int((sub.score > r.score).sum() + 1)
        print(f"{n} | {r.role} | {place} из {len(sub)} | оценка {r.score:.0f}")
    else:
        print(n, "| нет в таблице")

print("\n2) связь оценки игры с xG на 90 минут (Спирмен), по ролям:")
for role in roles:
    s = res[res.role == role]
    print(role, round(s[["score", "xg"]].corr(method="spearman").iloc[0, 1], 2))

print("\n3) клубы в прежних списках «недооценённых» (топ-8 по ролям, 1500+ мин):")
early = pd.concat([res[(res.role == r) & (res.mins >= 1500)].sort_values("resid").head(8)
                   for r in roles])
print(early.team.value_counts().head(8).to_string())

print("\n4) исправленные кандидаты (топ-30% по оценке в роли, возраст до 28, 1500+ мин, без late):")
ok = res[(res.score_pct >= 0.70) & (res.age <= 28) & (res.mins >= 1500) & ~(res.late == True)]
final = []
for role in roles:
    s = ok[ok.role == role].sort_values("resid").head(6)
    final.append(s)
    print(f"\n{role} (подходящих в роли: {(ok.role == role).sum()})")
    print(pd.DataFrame({
        "team": s.team, "age": s.age, "score": s.score.round(0),
        "value_m": s.value_m.round(1), "expected_m": s.expected_m.round(1),
        "discount_%": ((1 - s.value_m / s.expected_m) * 100).round(0),
    }).to_string())

final = pd.concat(final)
print("\nклубы в исправленных списках:")
print(final.team.value_counts().head(8).to_string())

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

folder = "/content/drive/MyDrive/football_portfolio"
merged4 = pd.read_pickle(f"{folder}/all_1516_merged.pkl")
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]

spec_v1 = {
    "ST":   {"xg": 1, "shots": 1, "key_passes": 1, "dribbles_won": 1,
             "prog_carries": 1, "pressures": 1, "fouls_won": 1},
    "AM_W": {"xg": 1, "key_passes": 1, "dribbles_won": 1, "prog_carries": 1,
             "prog_passes": 1, "shots": 1, "pressures": 1, "recoveries": 1},
    "CM":   {"prog_passes": 1, "key_passes": 1, "final_third_passes": 1, "pass_pct": 1,
             "prog_carries": 1, "tackles": 1, "interceptions": 1, "recoveries": 1, "pressures": 1},
    "DM":   {"prog_passes": 1, "pass_pct": 1, "long_passes": 1, "tackles": 1,
             "interceptions": 1, "recoveries": 1, "pressures": 1, "dispossessed": -1},
    "FB":   {"prog_carries": 1, "prog_passes": 1, "crosses": 1, "key_passes": 1,
             "dribbles_won": 1, "tackles": 1, "interceptions": 1, "recoveries": 1},
    "CB":   {"prog_passes": 1, "long_passes": 1, "pass_pct": 1, "interceptions": 1,
             "tackles": 1, "clearances": 1, "blocks": 1, "recoveries": 1,
             "aerial_lost": -1, "fouls": -1},
}
spec_v2 = dict(spec_v1)
spec_v2["ST"] = {"xg": 3, "shots": 1.5, "key_passes": 1, "dribbles_won": 1,
                 "prog_carries": 1, "pressures": 0.5}
spec_v2["AM_W"] = {"xg": 2, "key_passes": 2, "dribbles_won": 1, "prog_carries": 1,
                   "prog_passes": 1, "shots": 1, "pressures": 0.5, "recoveries": 0.5}

def run(spec):
    d = merged4.copy()
    d["score"] = np.nan
    for role, metrics in spec.items():
        idx = d.role == role
        num, den = 0, 0
        for m, w in metrics.items():
            pct = d.loc[idx, m].rank(pct=True)
            pct = pct if w > 0 else 1 - pct
            num = num + pct * abs(w)
            den += abs(w)
        d.loc[idx, "score"] = num / den * 100
    d["score_pct"] = d.groupby("role").score.rank(pct=True)
    d = d[d.value_eur.notna()].copy()
    d["log_value"] = np.log(d.value_eur)
    d["value_m"] = d.value_eur / 1e6
    out = []
    for role in roles:
        s = d[d.role == role].copy()
        mod = smf.ols("log_value ~ score + age + I(age**2) + C(league)", data=s).fit()
        s["expected_m"] = np.exp(mod.fittedvalues) / 1e6
        s["resid"] = mod.resid
        s["r2"] = mod.rsquared
        out.append(s)
    return pd.concat(out)

def candidates(res, top=10):
    ok = res[(res.score_pct >= 0.70) & (res.age <= 28) & (res.mins >= 1500) & ~(res.late == True)]
    return pd.concat([ok[ok.role == r].sort_values("resid").head(top) for r in roles])

r1, r2 = run(spec_v1), run(spec_v2)

print("1) звёзды в своей роли: версия 1 -> версия 2 (место из скольких):")
stars = ["Lionel Andrés Messi Cuccittini", "Cristiano Ronaldo dos Santos Aveiro",
         "Neymar da Silva Santos Junior", "Luis Alberto Suárez Díaz", "Gonzalo Gerardo Higuaín",
         "Zlatan Ibrahimović", "Sergio Leonel Agüero del Castillo", "Harry Kane", "Jamie Vardy",
         "Antoine Griezmann", "Edinson Roberto Cavani Gómez", "Paulo Bruno Exequiel Dybala"]
def place(res, n):
    if n not in res.index:
        return "нет"
    r = res.loc[n]
    return f"{int((res[res.role == r.role].score > r.score).sum() + 1)}/{(res.role == r.role).sum()}"
for n in stars:
    print(f"{n[:34]:35s} v1: {place(r1, n):9s} v2: {place(r2, n)}")

print("\n2) корреляция оценки с xG (Спирмен):")
for role in ["ST", "AM_W"]:
    for name, res in [("v1", r1), ("v2", r2)]:
        s = res[res.role == role]
        print(role, name, round(s[["score", "xg"]].corr(method="spearman").iloc[0, 1], 2))

print("\n3) R² моделей цены (v2):",
      {r: round(float(r2[r2.role == r].r2.iloc[0]), 2) for r in roles})

c1, c2 = candidates(r1), candidates(r2)
both = sorted(set(c1.index) & set(c2.index))
print(f"\n4) кандидатов в v1: {len(c1)}, в v2: {len(c2)}, в обеих: {len(both)}")

print("\nкандидаты версии 2 (топ-6 по роли), отметка * = есть и в версии 1:")
for role in roles:
    s = c2[c2.role == role].head(6)
    print(f"\n{role}")
    print(pd.DataFrame({
        "team": s.team, "age": s.age, "score": s.score.round(0),
        "value_m": s.value_m.round(1), "expected_m": s.expected_m.round(1),
        "both": ["*" if n in both else "" for n in s.index],
    }).to_string())

res_v2 = r2.copy()
res_v2.to_pickle(f"{folder}/all_1516_scored_v2.pkl")
print("\nсохранено: all_1516_scored_v2.pkl")

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import statsmodels.formula.api as smf
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
con = duckdb.connect(f"{folder}/transfermarkt-datasets.duckdb", read_only=True)

res = pd.read_pickle(f"{folder}/all_1516_scored_v2.pkl")
res = res[~(res.late == True)].copy()
res["tm_id"] = res.tm_id.astype(int)
res["logv"] = np.log(res.value_eur)
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]

# кандидаты версии 2 (те же правила, что и раньше)
ok = res[(res.score_pct >= 0.70) & (res.age <= 28) & (res.mins >= 1500)]
cand = pd.concat([ok[ok.role == r].sort_values("resid").head(10) for r in roles])
res["cand"] = res.index.isin(cand.index)
print("игроков в тесте:", len(res), "| кандидатов:", int(res.cand.sum()))

# оценки Transfermarkt после сезона
ids = ",".join(str(i) for i in res.tm_id.unique())
fut = con.sql(f"""
    SELECT player_id AS tm_id, date, market_value_in_eur AS v
    FROM player_valuations
    WHERE player_id IN ({ids})
      AND date >= DATE '2016-07-01' AND date <= DATE '2020-01-31'
      AND market_value_in_eur IS NOT NULL AND market_value_in_eur > 0
""").df()
fut["date"] = pd.to_datetime(fut.date)

horizons = {12: pd.Timestamp("2017-06-30"), 24: pd.Timestamp("2018-06-30"), 36: pd.Timestamp("2019-06-30")}
for h, t in horizons.items():
    w = fut[(fut.date >= t - pd.Timedelta(days=150)) & (fut.date <= t + pd.Timedelta(days=30))].copy()
    w["gap"] = (w.date - t).abs()
    w = w.sort_values("gap").groupby("tm_id").head(1).set_index("tm_id").v
    res[f"v{h}"] = res.tm_id.map(w)
    res[f"d{h}"] = np.log(res[f"v{h}"] / res.value_eur)

print("\nдоля игроков с оценкой через N месяцев:")
for h in horizons:
    print(f"{h} мес: все {res[f'v{h}'].notna().mean()*100:.0f}% | кандидаты {res[res.cand][f'v{h}'].notna().mean()*100:.0f}%")

print("\n1) регрессия на всех игроках: рост стоимости ~ оценка игры (+ стартовая стоимость, возраст, роль, лига)")
for h in horizons:
    s = res[res[f"d{h}"].notna()].copy()
    mod = smf.ols(f"d{h} ~ score + logv + age + I(age**2) + C(role) + C(league)", data=s).fit(cov_type="HC3")
    b = mod.params["score"] * 10
    lo, hi = mod.conf_int().loc["score"].values * 10
    print(f"{h} мес: n={len(s)}, +10 пунктов оценки -> {(np.exp(b)-1)*100:+.0f}% к росту стоимости "
          f"(95% ДИ {(np.exp(lo)-1)*100:+.0f}% … {(np.exp(hi)-1)*100:+.0f}%)")

print("\n2) кандидаты против похожих по роли, возрасту и стартовой стоимости:")
rng = np.random.default_rng(0)
for h in horizons:
    col = f"d{h}"
    pool = res[res[col].notna() & ~res.cand]
    cs = res[res.cand & res[col].notna()]
    diffs, nomatch = [], 0
    for name, r in cs.iterrows():
        m = pool[(pool.role == r.role) & ((pool.age - r.age).abs() <= 2) & ((pool.logv - r.logv).abs() <= 0.35)]
        if len(m) < 5:
            nomatch += 1
            continue
        diffs.append(r[col] - m[col].mean())
    diffs = np.array(diffs)
    boots = [rng.choice(diffs, len(diffs)).mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{h} мес: кандидатов с данными {len(cs)}, без пары в контроле {nomatch}, "
          f"разница в росте {(np.exp(diffs.mean())-1)*100:+.0f}% (95% ДИ {(np.exp(lo)-1)*100:+.0f}% … {(np.exp(hi)-1)*100:+.0f}%)")

print("\n3) что стало с кандидатами (млн евро):")
t = res[res.cand].copy()
print(pd.DataFrame({
    "team": t.team, "role": t.role, "age": t.age,
    "start": (t.value_eur / 1e6).round(1),
    "m12": (t.v12 / 1e6).round(1), "m24": (t.v24 / 1e6).round(1), "m36": (t.v36 / 1e6).round(1),
    "x24": (t.v24 / t.value_eur).round(1),
}).sort_values("x24", ascending=False).to_string())

res.to_pickle(f"{folder}/all_1516_backtest.pkl")

In [ ]:
import numpy as np
import pandas as pd

folder = "/content/drive/MyDrive/football_portfolio"
res = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
rng = np.random.default_rng(1)
DOWN = np.log(0.8)

def matched(cands, pool, col):
    rows = []
    for name, r in cands.iterrows():
        m = pool[(pool.role == r.role) & ((pool.age - r.age).abs() <= 2)
                 & ((pool.logv - r.logv).abs() <= 0.35)]
        m = m[m.index != name]
        if len(m) < 5:
            continue
        rows.append((name, r.role, r[col] - m[col].mean(),
                     float(r[col] > 0), (m[col] > 0).mean(),
                     float(r[col] < DOWN), (m[col] < DOWN).mean(), m[col].median()))
    return pd.DataFrame(rows, columns=["player", "role", "diff", "c_up", "k_up",
                                       "c_down", "k_down", "k_med"]).set_index("player")

def pct(x):
    return f"{(np.exp(x) - 1) * 100:+.0f}%"

print("1) среднее, медиана, доля выросших и упавших больше чем на 20%:")
for h in (12, 24, 36):
    col = f"d{h}"
    pool = res[res[col].notna() & ~res.cand]
    cs = res[res.cand & res[col].notna()]
    md = matched(cs, pool, col)
    print(f"{h} мес: n={len(md)} | средняя разница {pct(md['diff'].mean())}, "
          f"медианная {pct(md['diff'].median())} | выросли: кандидаты {md.c_up.mean()*100:.0f}% "
          f"vs контроль {md.k_up.mean()*100:.0f}% | упали >20%: кандидаты {md.c_down.mean()*100:.0f}% "
          f"vs контроль {md.k_down.mean()*100:.0f}%")

print("\n2) без пяти самых удачных кандидатов:")
for h in (12, 24, 36):
    col = f"d{h}"
    pool = res[res[col].notna() & ~res.cand]
    md = matched(res[res.cand & res[col].notna()], pool, col).sort_values("diff", ascending=False)
    print(f"{h} мес: всего {pct(md['diff'].mean())} -> без топ-5 {pct(md['diff'].iloc[5:].mean())} "
          f"(убраны: {', '.join(md.index[:5])})")

print("\n3) по ролям, 24 месяца:")
col = "d24"
pool = res[res[col].notna() & ~res.cand]
md = matched(res[res.cand & res[col].notna()], pool, col)
print(md.groupby("role").agg(n=("diff", "size"),
                             mean=("diff", lambda x: round((np.exp(x.mean()) - 1) * 100)),
                             median=("diff", lambda x: round((np.exp(x.median()) - 1) * 100)),
                             up=("c_up", "mean")).round(2).to_string())

print("\n4) контрольный эксперимент: случайные подходящие по возрасту и минутам игроки (24 мес)")
elig = res[(res.age <= 28) & (res.mins >= 1500) & ~res.cand & res[col].notna()]
need = res[res.cand].role.value_counts().to_dict()
real = md["diff"].mean()
sims = []
for _ in range(200):
    pick = pd.concat([elig[elig.role == r].sample(min(n, (elig.role == r).sum()),
                                                  random_state=int(rng.integers(1e9)))
                      for r, n in need.items()])
    pl = res[res[col].notna() & ~res.cand & ~res.index.isin(pick.index)]
    d = matched(pick, pl, col)["diff"].mean()
    sims.append(d)
sims = np.array(sims)
print(f"реальные кандидаты: {pct(real)} | случайные выборки: среднее {pct(sims.mean())}, "
      f"95-й процентиль {pct(np.percentile(sims, 95))}, максимум {pct(sims.max())}")
print(f"доля случайных выборок не хуже реальных: {(sims >= real).mean() * 100:.1f}%")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

folder = "/content/drive/MyDrive/football_portfolio"
res = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
rng = np.random.default_rng(2)
elig = (res.age <= 28) & (res.mins >= 1500)

def matched(cands, pool, col):
    rows = []
    for name, r in cands.iterrows():
        m = pool[(pool.role == r.role) & ((pool.age - r.age).abs() <= 2)
                 & ((pool.logv - r.logv).abs() <= 0.35)]
        if len(m) < 5:
            continue
        rows.append((name, r.role, r[col] - m[col].mean(), float(r[col] > 0), (m[col] > 0).mean()))
    return pd.DataFrame(rows, columns=["player", "role", "diff", "c_up", "k_up"]).set_index("player")

def pct(x):
    return f"{(np.exp(x) - 1) * 100:+.0f}%"

def boot(x, n=3000):
    b = [rng.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])

print("1) кандидаты против контроля из тех же условий (возраст ≤ 28, 1500+ минут, не выбраны моделью):")
mds = {}
for h in (12, 24, 36):
    col = f"d{h}"
    pool = res[elig & res[col].notna() & ~res.cand]
    md = matched(res[res.cand & res[col].notna()], pool, col)
    mds[h] = md
    lo, hi = boot(md["diff"].values)
    s = md["diff"].sort_values()
    print(f"{h} мес: n={len(md)} | разница {pct(md['diff'].mean())} (95% ДИ {pct(lo)} … {pct(hi)}) | "
          f"медиана {pct(md['diff'].median())} | без 5 лучших и 5 худших {pct(s.iloc[5:-5].mean())} | "
          f"выросли: {md.c_up.mean()*100:.0f}% vs {md.k_up.mean()*100:.0f}%")

print("\n2) по ролям, 24 месяца (в каждой роли около 10 игроков, это только наблюдение):")
md = mds[24]
print(md.groupby("role").agg(n=("diff", "size"),
                             mean=("diff", lambda x: round((np.exp(x.mean()) - 1) * 100)),
                             median=("diff", lambda x: round((np.exp(x.median()) - 1) * 100)),
                             share_up=("c_up", "mean")).round(2).to_string())

print("\n3) регрессия на всех игроках: рост стоимости ~ разрыв между ценой и ожидаемой ценой")
print("   (resid = log факт. стоимости минус log ожидаемой; -0.5 означает «цена примерно на 40% ниже ожидаемой»)")
for h in (12, 24, 36):
    s = res[res[f"d{h}"].notna()].copy()
    mod = smf.ols(f"d{h} ~ resid + score + age + I(age**2) + C(role) + C(league)", data=s).fit(cov_type="HC3")
    b = -0.5 * mod.params["resid"]
    lo, hi = sorted(-0.5 * mod.conf_int().loc["resid"].values)
    print(f"{h} мес: n={len(s)}, при resid = -0.5: {pct(b)} к росту стоимости "
          f"(95% ДИ {pct(lo)} … {pct(hi)}), p={mod.pvalues['resid']:.3f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

folder = "/content/drive/MyDrive/football_portfolio"
res = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
rng = np.random.default_rng(3)
elig = (res.age <= 28) & (res.mins >= 1500)
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]

def matched(cands, pool, col):
    rows = []
    for name, r in cands.iterrows():
        m = pool[(pool.role == r.role) & ((pool.age - r.age).abs() <= 2)
                 & ((pool.logv - r.logv).abs() <= 0.35)]
        m = m[m.index != name]
        if len(m) < 5:
            continue
        rows.append((name, r[col] - m[col].mean()))
    return pd.Series(dict(rows))

def pct(x):
    return (np.exp(x) - 1) * 100

need = res[res.cand].role.value_counts().to_dict()
stats = {}
print("плацебо с честным контролем (150 случайных выборок того же состава):")
for h in (12, 24, 36):
    col = f"d{h}"
    pool = res[elig & res[col].notna() & ~res.cand]
    real = matched(res[res.cand & res[col].notna()], pool, col)
    lo, hi = np.percentile([rng.choice(real.values, len(real)).mean() for _ in range(3000)], [2.5, 97.5])
    stats[h] = (real.mean(), lo, hi)
    sims = []
    for _ in range(150):
        pick = pd.concat([pool[pool.role == r].sample(min(n, int((pool.role == r).sum())),
                                                      random_state=int(rng.integers(1e9)))
                          for r, n in need.items()])
        pl = pool[~pool.index.isin(pick.index)]
        sims.append(matched(pick, pl, col).mean())
    sims = np.array(sims)
    print(f"{h} мес: реальные кандидаты {pct(real.mean()):+.0f}% | случайные: среднее {pct(sims.mean()):+.0f}%, "
          f"95-й процентиль {pct(np.percentile(sims, 95)):+.0f}% | "
          f"доля случайных не хуже реальных: {(sims >= real.mean()).mean() * 100:.1f}%")

# график для README
hs = [12, 24, 36]
vals = [pct(stats[h][0]) for h in hs]
err = [[v - pct(stats[h][1]) for v, h in zip(vals, hs)],
       [pct(stats[h][2]) - v for v, h in zip(vals, hs)]]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.bar([f"{h} months" for h in hs], vals, yerr=err, capsize=6, color="#2a6f97")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Extra growth in market value vs matched control (%)")
ax.set_title("Model-selected players vs matched regular starters\n(Top-4 leagues 2015/16, 95% bootstrap CI)", fontsize=10)
plt.tight_layout()
plt.savefig("backtest_candidates_vs_control.png", dpi=200, bbox_inches="tight")
plt.show()

# таблица кандидатов с итогами (включая неудачи): топ-3 в каждой роли по остатку модели
t = res[res.cand].copy()
top3 = pd.concat([t[t.role == r].sort_values("resid").head(3) for r in roles])
out = pd.DataFrame({
    "role": top3.role, "team": top3.team, "age": top3.age, "score": top3.score.round(0),
    "value_2016_m": (top3.value_eur / 1e6).round(1), "expected_m": top3.expected_m.round(1),
    "value_2017_m": (top3.v12 / 1e6).round(1), "value_2018_m": (top3.v24 / 1e6).round(1),
    "value_2019_m": (top3.v36 / 1e6).round(1),
})
print("\nтоп-3 кандидата в каждой роли по версии модели (в том числе неудачи):")
print(out.to_string())
out.to_csv("top3_candidates_by_role.csv")
t[["role", "team", "age", "score", "value_m", "expected_m", "v12", "v24", "v36"]].to_csv("all_candidates.csv")
print("\nфайлы сохранены в рабочей папке Colab: backtest_candidates_vs_control.png, top3_candidates_by_role.csv, all_candidates.csv")

In [ ]:
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
leagues = {"seriea": "Serie A", "premier": "Premier League", "laliga": "La Liga", "ligue1": "Ligue 1"}

# 1) доля передач команды в её матчах
share = {}
for key, name in leagues.items():
    ev = pd.read_pickle(f"{folder}/{key}_1516_events.pkl")
    pp = ev.loc[ev.type == "Pass", ["match_id", "team"]]
    del ev
    pp = pp.groupby(["match_id", "team"]).size().rename("p").reset_index()
    pp["opp"] = pp.groupby("match_id").p.transform("sum") - pp.p
    g = pp.groupby("team")[["p", "opp"]].sum()
    for t, v in (g.p / (g.p + g.opp)).items():
        share[(name, t)] = v
poss = pd.Series(share)
poss.to_pickle(f"{folder}/team_possession_1516.pkl")

# 2) данные игроков и поправка
d = pd.read_pickle(f"{folder}/all_1516_merged.pkl")
d["poss"] = [poss.get((l, t), np.nan) for l, t in zip(d.league, d.team)]
print("игроков без доли владения:", int(d.poss.isna().sum()))
factor = 0.5 / (1 - d.poss)
for m in ["tackles", "interceptions", "recoveries", "pressures", "clearances", "blocks"]:
    d[m + "_adj"] = d[m] * factor

shots_total = d.shots * d.mins / 90
d["xg_per_shot"] = (d.xg_total / shots_total).replace([np.inf, -np.inf], np.nan)
d["xg_per_shot"] = d.xg_per_shot.fillna(d.groupby("role").xg_per_shot.transform("median"))
d["gxg"] = (d.goals_total - d.xg_total) / d.mins * 90

# 3) измерения по ролям ("-" в начале означает, что чем меньше, тем лучше)
spec = {
    "ST": {
        "Threat": ["xg", "shots", "xg_per_shot"],
        "Creation": ["key_passes", "through_balls"],
        "Carrying": ["dribbles_won", "prog_carries"],
        "Linkup": ["passes", "pass_pct", "prog_passes"],
        "Workrate": ["pressures_adj", "recoveries_adj"],
    },
    "AM_W": {
        "Threat": ["xg", "shots", "xg_per_shot"],
        "Creation": ["key_passes", "through_balls", "crosses"],
        "Carrying": ["dribbles_won", "prog_carries"],
        "Progression": ["prog_passes", "final_third_passes"],
        "Workrate": ["pressures_adj", "recoveries_adj"],
    },
    "CM": {
        "Progression": ["prog_passes", "prog_carries", "final_third_passes"],
        "Creation": ["key_passes", "through_balls"],
        "Security": ["pass_pct", "passes", "-dispossessed"],
        "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
        "Pressing": ["pressures_adj"],
        "Threat": ["xg", "shots"],
    },
    "DM": {
        "Progression": ["prog_passes", "long_passes", "prog_carries"],
        "Security": ["pass_pct", "passes", "-dispossessed"],
        "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
        "Pressing": ["pressures_adj"],
        "Creation": ["key_passes", "final_third_passes"],
    },
    "FB": {
        "Attacking": ["crosses", "key_passes", "dribbles_won"],
        "Progression": ["prog_carries", "prog_passes"],
        "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
        "Security": ["pass_pct", "-dispossessed"],
        "Clearing": ["clearances_adj", "blocks_adj"],
    },
    "CB": {
        "Progression": ["prog_passes", "long_passes", "pass_pct"],
        "Defending": ["tackles_adj", "interceptions_adj", "blocks_adj"],
        "Clearing": ["clearances_adj"],
        "Recovery": ["recoveries_adj"],
        "Discipline": ["-aerial_lost", "-fouls"],
    },
}

dim_cols = {}
for role, dims in spec.items():
    idx = d.role == role
    for dim, mets in dims.items():
        parts = []
        for m in mets:
            col = m.lstrip("-")
            pct = d.loc[idx, col].rank(pct=True)
            parts.append(1 - pct if m.startswith("-") else pct)
        d.loc[idx, "dim_" + dim] = pd.concat(parts, axis=1).mean(axis=1) * 100
        dim_cols.setdefault(role, []).append("dim_" + dim)

d.to_pickle(f"{folder}/all_1516_dims.pkl")

# 4) проверки на здравый смысл
print("\nдоля передач команды: самые низкие и самые высокие")
tp = poss.sort_values()
print(tp.head(5).round(2).to_string())
print(tp.tail(5).round(2).to_string())

print("\nтоп-3 по каждому измерению в каждой роли:")
for role, cols in dim_cols.items():
    print(f"\n=== {role} ===")
    for c in cols:
        top = d[d.role == role].sort_values(c, ascending=False).head(3)
        print(f"{c[4:]:12s}", "; ".join(f"{n.split()[-1]} ({r.team}, {r[c]:.0f})" for n, r in top.iterrows()))

stars = ["Lionel Andrés Messi Cuccittini", "Luis Alberto Suárez Díaz", "Marco Verratti",
         "Carlos Henrique Casimiro", "Marcelo Vieira da Silva Júnior",
         "Thiago Emiliano da Silva", "Paulo Bruno Exequiel Dybala"]
print("\nпрофили известных игроков:")
for n in stars:
    if n in d.index:
        r = d.loc[n]
        print(f"{n} ({r.role}, {r.team}):",
              ", ".join(f"{c[4:]} {r[c]:.0f}" for c in dim_cols[r.role]))

In [ ]:
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

poss = pd.read_pickle(f"{folder}/team_possession_1516.pkl")
d = pd.read_pickle(f"{folder}/all_1516_merged.pkl")
d["poss"] = [poss.get((l, t), np.nan) for l, t in zip(d.league, d.team)]

def_mets = ["tackles", "interceptions", "recoveries", "pressures", "clearances", "blocks"]
base = 0.5 / (1 - d.poss)

# 1) как связана доля владения с защитной нагрузкой при разной силе поправки
print("k   | корреляция по 80 командам (доля владения vs средняя защитная нагрузка игроков)")
res_k = {}
for k in np.arange(0, 1.31, 0.1):
    f = base ** k
    load = (d[["tackles", "interceptions", "recoveries", "pressures"]].sum(axis=1) * f)
    tm = pd.DataFrame({"league": d.league, "team": d.team, "load": load, "poss": d.poss}) \
           .groupby(["league", "team"]).mean()
    res_k[round(k, 1)] = tm.load.corr(tm.poss)
    print(f"{k:.1f} | {res_k[round(k, 1)]:+.2f}")
k_best = min(res_k, key=lambda x: abs(res_k[x]))
print("\nвыбрано k =", k_best, "(корреляция", round(res_k[k_best], 2), ")")

factor = base ** k_best
for m in def_mets:
    d[m + "_adj"] = d[m] * factor

shots_total = d.shots * d.mins / 90
d["xg_per_shot"] = (d.xg_total / shots_total).replace([np.inf, -np.inf], np.nan)
d["xg_per_shot"] = d.xg_per_shot.fillna(d.groupby("role").xg_per_shot.transform("median"))
d["gxg"] = (d.goals_total - d.xg_total) / d.mins * 90

# 2) те же измерения, что и раньше
spec = {
    "ST": {"Threat": ["xg", "shots", "xg_per_shot"], "Creation": ["key_passes", "through_balls"],
           "Carrying": ["dribbles_won", "prog_carries"], "Linkup": ["passes", "pass_pct", "prog_passes"],
           "Workrate": ["pressures_adj", "recoveries_adj"]},
    "AM_W": {"Threat": ["xg", "shots", "xg_per_shot"], "Creation": ["key_passes", "through_balls", "crosses"],
             "Carrying": ["dribbles_won", "prog_carries"], "Progression": ["prog_passes", "final_third_passes"],
             "Workrate": ["pressures_adj", "recoveries_adj"]},
    "CM": {"Progression": ["prog_passes", "prog_carries", "final_third_passes"],
           "Creation": ["key_passes", "through_balls"], "Security": ["pass_pct", "passes", "-dispossessed"],
           "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
           "Pressing": ["pressures_adj"], "Threat": ["xg", "shots"]},
    "DM": {"Progression": ["prog_passes", "long_passes", "prog_carries"],
           "Security": ["pass_pct", "passes", "-dispossessed"],
           "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
           "Pressing": ["pressures_adj"], "Creation": ["key_passes", "final_third_passes"]},
    "FB": {"Attacking": ["crosses", "key_passes", "dribbles_won"], "Progression": ["prog_carries", "prog_passes"],
           "Defending": ["tackles_adj", "interceptions_adj", "recoveries_adj"],
           "Security": ["pass_pct", "-dispossessed"], "Clearing": ["clearances_adj", "blocks_adj"]},
    "CB": {"Progression": ["prog_passes", "long_passes", "pass_pct"],
           "Defending": ["tackles_adj", "interceptions_adj", "blocks_adj"],
           "Clearing": ["clearances_adj"], "Recovery": ["recoveries_adj"],
           "Discipline": ["-aerial_lost", "-fouls"]},
}
dim_cols = {}
for role, dims in spec.items():
    idx = d.role == role
    for dim, mets in dims.items():
        parts = []
        for m in mets:
            col = m.lstrip("-")
            pct = d.loc[idx, col].rank(pct=True)
            parts.append(1 - pct if m.startswith("-") else pct)
        d.loc[idx, "dim_" + dim] = pd.concat(parts, axis=1).mean(axis=1) * 100
        dim_cols.setdefault(role, []).append("dim_" + dim)

d.attrs["k_best"] = float(k_best)
d.to_pickle(f"{folder}/all_1516_dims.pkl")

# 3) проверка: защитные измерения, лидеры и доля «ПСЖ» в топ-10
print("\nлидеры защитных измерений после поправки:")
for role, dim in [("FB", "Defending"), ("CM", "Pressing"), ("CM", "Defending"), ("DM", "Pressing"),
                  ("CB", "Defending"), ("CB", "Recovery")]:
    t = d[d.role == role].sort_values("dim_" + dim, ascending=False).head(5)
    print(f"{role} {dim}:", "; ".join(f"{n.split()[-1]} ({r.team}, {r['dim_' + dim]:.0f})" for n, r in t.iterrows()))

print("\nпрофили:")
for n in ["Marco Verratti", "Thiago Emiliano da Silva", "Carlos Henrique Casimiro", "Francis Coquelin"]:
    if n in d.index:
        r = d.loc[n]
        print(f"{n} ({r.role}, {r.team}):", ", ".join(f"{c[4:]} {r[c]:.0f}" for c in dim_cols[r.role]))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
dims = pd.read_pickle(f"{folder}/all_1516_dims.pkl")
dcols = [c for c in dims.columns if c.startswith("dim_")]
bt = bt.drop(columns=[c for c in dcols if c in bt.columns]).join(dims[dcols])
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]

def pct(b):
    return (np.exp(b) - 1) * 100

def fit(formula, data, name):
    m = smf.ols(formula, data=data).fit(cov_type="HC3")
    ci = m.conf_int()
    out = {}
    for c in m.params.index:
        if c.startswith("dim_"):
            lo, hi = pct(ci.loc[c, 0]), pct(ci.loc[c, 1])
            star = "*" if (lo > 0 or hi < 0) else " "
            out[c] = (pct(m.params[c]), lo, hi, star)
    return out, m.rsquared

for role in roles:
    s = bt[bt.role == role].copy()
    cols = [c for c in dcols if s[c].notna().all()]
    for c in cols:
        s[c] = (s[c] - s[c].mean()) / s[c].std()
    terms = " + ".join(cols)
    cm = s[cols].corr().abs().values
    np.fill_diagonal(cm, 0)

    price, r2p = fit(f"logv ~ {terms} + age + I(age**2) + C(league)", s, "price")
    g = {}
    for h in (24, 36):
        sh = s[s[f"d{h}"].notna()]
        g[h], _ = fit(f"d{h} ~ {terms} + logv + age + I(age**2) + C(league)", sh, f"d{h}")

    print(f"\n=== {role} (n={len(s)}, R² цены {r2p:.2f}, макс. корреляция между измерениями {cm.max():.2f}) ===")
    print("измерение     |  цена, % на +1 SD [95% ДИ]    | рост за 24 мес               | рост за 36 мес")
    for c in cols:
        p, g24, g36 = price[c], g[24][c], g[36][c]
        print(f"{c[4:]:13s} | {p[0]:+5.0f}% [{p[1]:+4.0f}, {p[2]:+4.0f}]{p[3]} | "
              f"{g24[0]:+5.0f}% [{g24[1]:+4.0f}, {g24[2]:+4.0f}]{g24[3]} | "
              f"{g36[0]:+5.0f}% [{g36[1]:+4.0f}, {g36[2]:+4.0f}]{g36[3]}")
print("\n* означает, что интервал не включает ноль")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
dims = pd.read_pickle(f"{folder}/all_1516_dims.pkl")
dcols = [c for c in dims.columns if c.startswith("dim_")]
bt = bt.drop(columns=[c for c in dcols if c in bt.columns]).join(dims[dcols])
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
rng = np.random.default_rng(5)

print("пропусков в возрасте:", int(bt.age.isna().sum()), "| в logv:", int(bt.logv.isna().sum()))

splits = [(["Premier League", "La Liga"], ["Serie A", "Ligue 1"]),
          (["Serie A", "Ligue 1"], ["Premier League", "La Liga"])]

def design(df, cols, mu=None, sd=None):
    X = df[cols].copy()
    X["logv"] = df.logv
    X["age"] = df.age
    X["age2"] = df.age ** 2
    if mu is None:
        mu, sd = X.mean(), X.std().replace(0, 1)
    return (X - mu) / sd, mu, sd

picked, coefs = [], {}
for si, (tr_l, te_l) in enumerate(splits):
    for role in roles:
        s = bt[bt.role == role]
        cols = [c for c in dcols if s[c].notna().all()]
        need = cols + ["logv", "age"]
        tr = s[s.league.isin(tr_l) & s.d24.notna() & s.d36.notna()].dropna(subset=need)
        te = s[s.league.isin(te_l)].dropna(subset=need).copy()
        y = (tr.d24 + tr.d36) / 2
        Xtr, mu, sd = design(tr, cols)
        mod = RidgeCV(alphas=[1, 3, 10, 30, 100, 300]).fit(Xtr, y)
        coefs[(si, role)] = pd.Series(mod.coef_[:len(cols)], index=[c[4:] for c in cols])
        Xte, _, _ = design(te, cols, mu, sd)
        te["pred"] = mod.predict(Xte)
        el = te[(te.age <= 28) & (te.mins >= 1500)]
        picked.append(el.sort_values("pred", ascending=False).head(10).assign(split=si))
cand = pd.concat(picked)
bt["is_c"] = bt.index.isin(cand.index)
elig = (bt.age <= 28) & (bt.mins >= 1500)

def pct(x):
    return (np.exp(x) - 1) * 100

def matched(cands, pool, col):
    rows = []
    for name, r in cands.iterrows():
        m = pool[(pool.role == r.role) & ((pool.age - r.age).abs() <= 2)
                 & ((pool.logv - r.logv).abs() <= 0.35)]
        m = m[m.index != name]
        if len(m) < 5:
            continue
        rows.append((name, r.role, r[col] - m[col].mean(), float(r[col] > 0), (m[col] > 0).mean()))
    return pd.DataFrame(rows, columns=["player", "role", "diff", "c_up", "k_up"]).set_index("player")

def evaluate(h):
    col = f"d{h}"
    parts = []
    for si, (tr_l, te_l) in enumerate(splits):
        pool = bt[elig & bt.league.isin(te_l) & bt[col].notna() & ~bt.is_c]
        cs = cand[(cand.split == si) & cand[col].notna()]
        parts.append(matched(cs, pool, col))
    return pd.concat(parts)

def placebo(h, n=100):
    col = f"d{h}"
    out = []
    for _ in range(n):
        parts = []
        for si, (tr_l, te_l) in enumerate(splits):
            base = bt[elig & bt.league.isin(te_l) & bt[col].notna() & ~bt.is_c]
            pick = pd.concat([base[base.role == r].sample(min(10, int((base.role == r).sum())),
                                                           random_state=int(rng.integers(1e9)))
                              for r in roles])
            pool = base[~base.index.isin(pick.index)]
            parts.append(matched(pick, pool, col))
        out.append(pd.concat(parts)["diff"].mean())
    return np.array(out)

print("1) воспроизводимость: коэффициенты качеств (ridge) в двух независимых половинах")
print("   A обучена на Premier League + La Liga, B на Serie A + Ligue 1")
agree, total = 0, 0
for role in roles:
    a, b = coefs[(0, role)], coefs[(1, role)]
    t = pd.DataFrame({"A": a.round(3), "B": b.round(3)})
    t["знак совпал"] = np.where(np.sign(a) == np.sign(b), "да", "нет")
    agree += int((t["знак совпал"] == "да").sum()); total += len(t)
    print(f"\n{role}\n{t.to_string()}")
print(f"\nзнак совпал у {agree} из {total} качеств")

print("\n2) отбор в «чужих» лигах (по 10 игроков на роль в каждой половине, всего", len(cand), "):")
for h in (12, 24, 36):
    md = evaluate(h)
    lo, hi = np.percentile([rng.choice(md["diff"].values, len(md)).mean() for _ in range(3000)], [2.5, 97.5])
    pl = placebo(h)
    print(f"{h} мес: n={len(md)} | разница {pct(md['diff'].mean()):+.0f}% (95% ДИ {pct(lo):+.0f}% … {pct(hi):+.0f}%) | "
          f"медиана {pct(md['diff'].median()):+.0f}% | выросли {md.c_up.mean()*100:.0f}% vs {md.k_up.mean()*100:.0f}% | "
          f"плацебо {pct(pl.mean()):+.0f}% (не хуже реальных в {(pl >= md['diff'].mean()).mean()*100:.0f}% случаев)")

md24 = evaluate(24)
print("\nпо ролям, 24 месяца:")
print(md24.groupby("role")["diff"].agg(n="size", mean=lambda x: round(pct(x.mean())),
                                        median=lambda x: round(pct(x.median()))).to_string())

print("\n3) отобранные игроки (топ-5 по предсказанному росту в каждой роли), млн евро:")
for role in roles:
    t = cand[cand.role == role].sort_values("pred", ascending=False).head(5)
    print(f"\n{role}")
    print(pd.DataFrame({"team": t.team, "league": t.league, "age": t.age,
                        "start": (t.value_eur / 1e6).round(1), "m12": (t.v12 / 1e6).round(1),
                        "m24": (t.v24 / 1e6).round(1), "m36": (t.v36 / 1e6).round(1),
                        "pred": t.pred.round(2)}).to_string())

cand[["role", "team", "league", "age", "value_eur", "v12", "v24", "v36", "pred", "split"]].to_csv(
    f"{folder}/oos_candidates.csv")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
cand = pd.read_csv(f"{folder}/oos_candidates.csv", index_col=0)
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
rng = np.random.default_rng(11)
held_out = [["Serie A", "Ligue 1"], ["Premier League", "La Liga"]]   # лиги, не участвовавшие в обучении модели

def pct(b):
    return (np.exp(b) - 1) * 100

def build(min_mins):
    parts = []
    for si, te_l in enumerate(held_out):
        s = bt[bt.league.isin(te_l) & (bt.age <= 28) & (bt.mins >= min_mins)].dropna(subset=["age", "logv"]).copy()
        s["cand"] = s.index.isin(cand[cand.split == si].index).astype(int)
        s["split"] = si
        parts.append(s)
    return pd.concat(parts)

FORMULA = "d{h} ~ cand + logv + age + I(age**2) + C(role) + C(league)"

def real_effect(df, h):
    s = df[df[f"d{h}"].notna()]
    m = smf.ols(FORMULA.format(h=h), data=s).fit(cov_type="HC3")
    return m.params["cand"], m.conf_int().loc["cand"].values, m.pvalues["cand"], int(s.cand.sum()), len(s)

def permuted(df, h, n=200):
    s = df[df[f"d{h}"].notna()].copy()
    counts = s[s.cand == 1].groupby(["split", "role"]).size()
    pools = {k: s[(s.split == k[0]) & (s.role == k[1]) & (s.mins >= 1500)].index.values for k in counts.index}
    out = []
    for _ in range(n):
        lab = pd.Series(0, index=s.index)
        for k, c in counts.items():
            lab.loc[rng.choice(pools[k], min(int(c), len(pools[k])), replace=False)] = 1
        s["cand"] = lab
        out.append(smf.ols(FORMULA.format(h=h), data=s).fit().params["cand"])
    return np.array(out)

for mm in (900, 1500):
    df = build(mm)
    print(f"\n=== контроль: игроки до 28 лет с {mm}+ минутами в «чужих» лигах "
          f"(всего {len(df)}, кандидатов {int(df.cand.sum())}) ===")
    print("средний возраст: кандидаты", round(df[df.cand == 1].age.mean(), 1),
          "| остальные", round(df[df.cand == 0].age.mean(), 1))
    for h in (12, 24, 36):
        b, ci, p, nc, n = real_effect(df, h)
        pl = permuted(df, h)
        print(f"{h} мес: кандидатов {nc} из {n} | доп. рост {pct(b):+.0f}% "
              f"(95% ДИ {pct(ci[0]):+.0f}% … {pct(ci[1]):+.0f}%), p={p:.3f} | "
              f"перестановки: среднее {pct(pl.mean()):+.0f}%, 95-й процентиль {pct(np.percentile(pl, 95)):+.0f}%, "
              f"не хуже реальных в {(pl >= b).mean() * 100:.1f}% случаев")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
dims = pd.read_pickle(f"{folder}/all_1516_dims.pkl")
dcols = [c for c in dims.columns if c.startswith("dim_")]
bt = bt.drop(columns=[c for c in dcols if c in bt.columns]).join(dims[dcols])
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
splits = [(["Premier League", "La Liga"], ["Serie A", "Ligue 1"]),
          (["Serie A", "Ligue 1"], ["Premier League", "La Liga"])]
rng = np.random.default_rng(21)

def feats(df, cols, with_dims):
    X = pd.DataFrame({"logv": df.logv, "age": df.age, "age2": df.age ** 2}, index=df.index)
    if with_dims:
        for c in cols:
            X[c] = df[c]
    return X

parts = []
for si, (tr_l, te_l) in enumerate(splits):
    for role in roles:
        s = bt[bt.role == role]
        cols = [c for c in dcols if s[c].notna().all()]
        need = cols + ["logv", "age"]
        ok = s.d24.notna() & s.d36.notna()
        tr = s[s.league.isin(tr_l) & ok].dropna(subset=need)
        te = s[s.league.isin(te_l) & ok].dropna(subset=need)
        ytr = (tr.d24 + tr.d36) / 2
        out = pd.DataFrame({"y": (te.d24 + te.d36) / 2, "role": role, "split": si,
                            "age": te.age, "mins": te.mins, "naive": ytr.mean()}, index=te.index)
        for name, wd in (("base", False), ("full", True)):
            Xtr, Xte = feats(tr, cols, wd), feats(te, cols, wd)
            mu, sd = Xtr.mean(), Xtr.std().replace(0, 1)
            mod = RidgeCV(alphas=[1, 3, 10, 30, 100, 300]).fit((Xtr - mu) / sd, ytr)
            out[name] = mod.predict((Xte - mu) / sd)
        parts.append(out)
res = pd.concat(parts)

def report(df, label):
    e_n = (df.y - df.naive) ** 2
    e_b = (df.y - df.base) ** 2
    e_f = (df.y - df.full) ** 2
    skill_b = 1 - e_b.sum() / e_n.sum()
    skill_f = 1 - e_f.sum() / e_n.sum()
    diff = (e_b - e_f).values
    boots = [rng.choice(diff, len(diff)).mean() for _ in range(3000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"\n{label}: n={len(df)}")
    print(f"  качество прогноза на новых лигах: только возраст+цена {skill_b:+.3f} | "
          f"с измерениями {skill_f:+.3f}")
    print(f"  выигрыш от измерений: средняя разница квадратов ошибок {diff.mean():+.4f} "
          f"(95% ДИ {lo:+.4f} … {hi:+.4f}), положительное значение = измерения помогают")
    g = df.copy()
    for m in ("base", "full"):
        g["top_" + m] = g.groupby(["split", "role"])[m].rank(pct=True) >= 0.8
    print(f"  средний рост за 24-36 мес у топ-20% по прогнозу: "
          f"возраст+цена {g[g.top_base].y.mean():+.2f} | с измерениями {g[g.top_full].y.mean():+.2f} "
          f"| все игроки {g.y.mean():+.2f} (log-единицы)")

report(res, "все игроки")
report(res[res.age <= 28], "игроки до 28 лет")
report(res[(res.age <= 28) & (res.mins >= 1500)], "до 28 лет и 1500+ минут")

print("\nпо ролям (все игроки), выигрыш от измерений:")
for role in roles:
    d = res[res.role == role]
    e_b = ((d.y - d.base) ** 2).sum()
    e_f = ((d.y - d.full) ** 2).sum()
    print(f"  {role}: n={len(d)}, ошибка снижается на {(1 - e_f / e_b) * 100:+.1f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
roles = ["ST", "AM_W", "CM", "DM", "FB", "CB"]
splits = [(["Premier League", "La Liga"], ["Serie A", "Ligue 1"]),
          (["Serie A", "Ligue 1"], ["Premier League", "La Liga"])]
rng = np.random.default_rng(31)

def X_of(df, with_score):
    X = pd.DataFrame({"logv": df.logv, "age": df.age, "age2": df.age ** 2}, index=df.index)
    if with_score:
        X["score"] = df.score
    return X

parts = []
for tr_l, te_l in splits:
    for role in roles:
        s = bt[bt.role == role]
        ok = s.d24.notna() & s.d36.notna()
        need = ["logv", "age", "score"]
        tr = s[s.league.isin(tr_l) & ok].dropna(subset=need)
        te = s[s.league.isin(te_l) & ok].dropna(subset=need)
        ytr = (tr.d24 + tr.d36) / 2
        out = pd.DataFrame({"y": (te.d24 + te.d36) / 2, "role": role, "age": te.age,
                            "mins": te.mins, "naive": ytr.mean()}, index=te.index)
        for name, ws in (("base", False), ("score", True)):
            mod = LinearRegression().fit(X_of(tr, ws), ytr)
            out[name] = mod.predict(X_of(te, ws))
        parts.append(out)
res = pd.concat(parts)

def report(df, label):
    e_n = (df.y - df.naive) ** 2
    e_b = (df.y - df.base) ** 2
    e_s = (df.y - df.score) ** 2
    diff = (e_b - e_s).values
    lo, hi = np.percentile([rng.choice(diff, len(diff)).mean() for _ in range(3000)], [2.5, 97.5])
    print(f"{label}: n={len(df)} | качество прогноза: возраст+цена {1 - e_b.sum() / e_n.sum():+.3f} "
          f"| плюс общая оценка {1 - e_s.sum() / e_n.sum():+.3f} | выигрыш {diff.mean():+.4f} "
          f"(95% ДИ {lo:+.4f} … {hi:+.4f})")

report(res, "все игроки")
report(res[res.age <= 28], "до 28 лет")
report(res[(res.age <= 28) & (res.mins >= 1500)], "до 28 лет, 1500+ минут")

b = LinearRegression().fit(X_of(bt.dropna(subset=["logv", "age", "score"]), True),
                           ((bt.d24 + bt.d36) / 2)[bt.dropna(subset=["logv", "age", "score"]).index].fillna(0))
print("\nпримечание: последняя строка (коэффициент) только для справки, не для выводов")

In [ ]:
import os, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/football_portfolio"
os.makedirs(f"{folder}/cards", exist_ok=True)

bt = pd.read_pickle(f"{folder}/all_1516_backtest.pkl")
dims = pd.read_pickle(f"{folder}/all_1516_dims.pkl")
dcols = [c for c in dims.columns if c.startswith("dim_")]
extra = ["goals_total", "xg_total", "assists_total"]
d = bt.drop(columns=[c for c in dcols + extra if c in bt.columns]).join(dims[dcols + extra])
cand = pd.read_csv(f"{folder}/oos_candidates.csv", index_col=0)

# все числовые столбцы приводим к числам
num_cols = dcols + extra + ["value_eur", "v12", "v24", "v36", "age", "mins"]
for c in num_cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")

N = 2
sel = cand.sort_values("pred", ascending=False).groupby("role").head(N)
order = {"ST": 0, "AM_W": 1, "CM": 2, "DM": 3, "FB": 4, "CB": 5}
sel = sel.assign(o=sel.role.map(order)).sort_values(["o", "pred"], ascending=[True, False])

def short(n):
    t = d.loc[n, "tm_name"]
    return t if isinstance(t, str) else n

def role_cols(role):
    pool = d[d.role == role]
    return [c for c in dcols if pool[c].notna().all()]

def similar(name, k=3):
    role = d.loc[name, "role"]
    cols = role_cols(role)
    pool = d[d.role == role][cols].astype(float)
    diff = pool.sub(pool.loc[name], axis=1)
    dist = np.sqrt((diff ** 2).mean(axis=1)).drop(index=name)
    return dist.sort_values().head(k)

def money(x):
    return f"{x / 1e6:.1f}" if pd.notna(x) else "n/a"

rows = []
for name in sel.index:
    r = d.loc[name]
    cols = role_cols(r.role)
    vals = r[cols].astype(float).values
    labels = [c[4:] for c in cols]
    strong = [l for l, v in sorted(zip(labels, vals), key=lambda x: -x[1]) if v >= 70][:2]
    weak = [l for l, v in sorted(zip(labels, vals), key=lambda x: x[1]) if v <= 30][:2]
    sim = similar(name)
    traj = " > ".join(money(x) for x in [r.value_eur, r.v12, r.v24, r.v36])

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.1, 1]})
    colors = ["#2a6f97" if v >= 50 else "#c97b63" for v in vals]
    ax.barh(labels[::-1], vals[::-1], color=colors[::-1])
    ax.axvline(50, color="gray", lw=0.8, ls="--")
    ax.set_xlim(0, 100)
    ax.set_xlabel("percentile within role (top-4 leagues, 2015/16)")
    ax.set_title(f"{short(name)} | {r.role} | {r.team}", fontsize=11, loc="left")

    ax2.axis("off")
    stat = f"Age {r.age:.0f} | {int(r.mins)} min | {r.league}"
    if r.role in ("ST", "AM_W"):
        stat += f"\nGoals {int(r.goals_total)} | xG {r.xg_total:.1f} | assists {int(r.assists_total)}"
    lines = [stat, "",
             "Strengths: " + (", ".join(strong) if strong else "none above the 70th percentile"),
             "Weaknesses: " + (", ".join(weak) if weak else "none below the 30th percentile"), "",
             "Most similar profiles (same role):"]
    for n in sim.index:
        lines.append(f"  {short(n)} ({d.loc[n, 'team']}, {money(d.loc[n, 'value_eur'])}M)")
    lines += ["", "Market value 2016 > 2017 > 2018 > 2019 (EUR M):", "  " + traj]
    ax2.text(0, 1, "\n".join(textwrap.fill(l, 58, subsequent_indent="    ") if len(l) > 58 else l for l in lines),
             va="top", fontsize=9.5, linespacing=1.4)

    plt.tight_layout()
    safe = "".join(ch for ch in short(name) if ch.isalnum() or ch in "-_ ").strip().replace(" ", "_")
    plt.savefig(f"{folder}/cards/{r.role}_{safe}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    rows.append({"role": r.role, "player": short(name), "team": r.team, "age": round(r.age, 1),
                 "strengths": ", ".join(strong), "weaknesses": ", ".join(weak),
                 "value_2016_m": money(r.value_eur), "2017": money(r.v12),
                 "2018": money(r.v24), "2019": money(r.v36)})

summary = pd.DataFrame(rows)
summary.to_csv(f"{folder}/cards/scouting_cards_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nкарточки сохранены в:", f"{folder}/cards")